<a href="https://colab.research.google.com/github/HNXJ/GSDR/blob/main/Biophys_SX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Biophysical modeling supporting material

Using genetic-stochastic delta rule for neuronal biophysical HH models in Jaxley (GSDR implemented on Jax/optax)


## Setup

### Packages

In [ ]:
%pip install jaxley

import os
os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=16'

from jax import config
config.update("jax_enable_x64", True)
config.update("jax_platform_name", "cpu") # 'gpu' / 'cpu'
# config.update("jax_debug_nans", True)

import jax
import numpy as np
import jaxley as jx
import jax.numpy as jnp
import matplotlib.pyplot as plt
import jaxley.optimize.transforms as jt

from jax import jit, vmap, value_and_grad
from jaxley.channels import Leak, HH
from jaxley.synapses import IonotropicSynapse
from jaxley.connect import fully_connect, sparse_connect, connect
from scipy import ndimage
import scipy
import pickle


figure_save_path = "/content/data/Figures"
model_save_path = "/content/data/NeuralCircuits"
save_dir = "/content/data/SignalArrays/"

# from google.colab import drive
# drive.mount('/content/drive/')
# figure_save_path = "/content/drive/MyDrive/Workspace/Figures" # Drive
# model_save_path = "/content/drive/MyDrive/Workspace/Models" # Drive
# save_dir = "/content/drive/MyDrive/Workspace/" # Drive


file_name = "selected_unit_spiking.npy"

save_path = os.path.join(save_dir, file_name)
selected_unit_spiking = jnp.load(save_path)
print(f"'selected_unit_spiking' loaded from {save_path}")
print(f"Shape of loaded_selected_unit_spiking: {selected_unit_spiking.shape}")

### Functions

In [ ]:
import jax.numpy as jnp
import jaxley as jx
import numpy as np
import jaxley.optimize.transforms as jt
import jax
import jax.scipy.signal

from jax import jit, vmap, value_and_grad
from jaxley.channels import Leak, HH, Channel
from jaxley.synapses import IonotropicSynapse, Synapse
from jaxley.connect import fully_connect, sparse_connect, connect

import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap # Import for custom colormap
from scipy import signal # Import scipy.signal for spectrogram
from scipy import ndimage # Import scipy.ndimage for smoothing

import optax
from flax.struct import dataclass # For GSDR
from typing import Any, Callable, NamedTuple, Optional, Tuple # Added Tuple
from scipy.ndimage import zoom, gaussian_filter

from scipy import signal
import matplotlib.pyplot as plt
import jax.numpy as jnp
import numpy as np


def extend_traces_for_spectrogram(traces, nperseg):
    """
    Mirrors the beginning and end of the traces based on the spectrogram window size.
    Input: traces (N, T)
    Output: traces_extended (N, T + nperseg)
    """
    half_window = int(nperseg // 2)
    # Mirror the beginning (reverse first half_window samples)
    prefix = traces[:, 1 : half_window + 1][:, ::-1]
    suffix = traces[:, -half_window - 1 : -1][:, ::-1]

    traces_extended = jnp.concatenate([prefix, traces, suffix], axis=1)
    return traces_extended

def compute_summed_spectrogram(traces_extended, fs, nperseg, noverlap):
    """
    Calculates spectrogram on each row separately and sums them up.
    """
    # Convert to numpy for scipy.signal
    traces_np = np.array(traces_extended)

    total_Sxx = None
    freqs = None
    times = None

    for i in range(traces_np.shape[0]):
        f, t, Sxx = signal.spectrogram(traces_np[i], fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap)
        if total_Sxx is None:
            total_Sxx = Sxx
            freqs = f
            times = t
        else:
            total_Sxx += Sxx

    temp_n = total_Sxx.shape[1]
    temp_w = int(temp_n/20)
    temp_d = int(temp_w/20)
    density_Sxx = np.zeros(total_Sxx.shape)

    for i in range(0, temp_n, max(1, temp_d)):
        temp_l = i
        temp_r = np.min([i + temp_w, temp_n])
        density_Sxx[:, temp_l:temp_r] = total_Sxx[:, temp_l:temp_r] / np.max(total_Sxx[:, temp_l:temp_r])

    return freqs, times, density_Sxx


def vis_smoothed_spectrogram(ax, data, t_range, f_range, zoom_fac=3, sigma=1.5):
    """
    Helper to upsample, smooth, and visualize the spectrogram image.
    """
    # Upsample
    data_up = zoom(data, zoom_fac, order=1)
    # Smooth
    data_smooth = gaussian_filter(data_up, sigma=sigma)

    # Plot
    # Note: imshow extent is [left, right, bottom, top]
    extent = [t_range[0], t_range[-1], f_range[0], f_range[1]]
    ax.imshow(data_smooth, aspect='auto', origin='lower', cmap='jet', extent=extent, interpolation='bicubic')


def test_net_vis_comp_psd(net, params, input_scalar, label_psd, label_psd_2, interval1=(1, 500), interval2=(501, 1000), savename=None):
    """
    Visualizes the PSD prediction and loss for a single input/label pair over two specific intervals.
    """
    # 1. Simulate (Scalar input)
    traces = simulate(params, input_scalar) # Shape (Cells, Time)

    # Helper to calculate PSD and stats for a specific interval
    def analyze_interval(traces_segment, dt):
        # PSD
        signal = jnp.mean(traces_segment, axis=0)
        N = signal.shape[-1]
        if N == 0:
            return jnp.zeros_like(global_psd_interval), 0.0

        fs = 1000.0 / dt
        signal_fft = jnp.fft.rfft(signal)
        freqs = jnp.fft.rfftfreq(N, d=dt/1000)
        psd_raw = jnp.abs(signal_fft)**2 / (N * fs)

        target_freqs = global_psd_interval
        interpolated_psd = jnp.interp(target_freqs, freqs, psd_raw)
        max_psd = jnp.max(interpolated_psd)
        prediction_psd = interpolated_psd / (max_psd + 1e-6)

        # Firing Rate
        threshold = -20.0
        spikes = (traces_segment[:, :-1] < threshold) & (traces_segment[:, 1:] >= threshold)
        count = jnp.sum(spikes)
        n_cells = traces_segment.shape[0]
        duration_sec = (traces_segment.shape[1] * dt) / 1000.0
        mean_fr = count / n_cells / duration_sec

        return prediction_psd, mean_fr

    # Indices for slicing
    idx_start1 = int(interval1[0] / dt_global)
    idx_end1 = int(interval1[1] / dt_global)
    idx_start2 = int(interval2[0] / dt_global)
    idx_end2 = int(interval2[1] / dt_global)

    # Analyze Intervals
    psd1, mean_fr1 = analyze_interval(traces[:, idx_start1:idx_end1], dt_global)
    psd2, mean_fr2 = analyze_interval(traces[:, idx_start2:idx_end2], dt_global)

    # Calculate Loss for Interval 1
    epsilon = 1e-6
    psd_loss1 = jnp.sum(jnp.square(label_psd * jnp.log((psd1 + epsilon) / (labels + epsilon))))
    psd_loss2 = jnp.sum(jnp.square(label_psd_2 * jnp.log((psd2 + epsilon) / (labels + epsilon))))

    # Firing rate penalty approximation for single scalar mean FR (assuming uniform distribution roughly)
    # Note: This differs slightly from calculating per neuron then averaging penalty, but fits the "mean_fr" scalar we have
    penalty_lower = jnp.exp(lower_c - mean_fr1)
    penalty_upper = jnp.exp(mean_fr1 - upper_c)
    firing_rate_penalty1 = penalty_lower + penalty_upper

    total_loss1 = psd_loss1 * psd_weight + firing_rate_weight * firing_rate_penalty1

    # Visualization
    fig, axs = plt.subplots(2, 1, figsize=(12, 6))

    # Upsample and smoothen prediction lines
    upsample_factor = 10
    smoothing_sigma = 1.5

    global_psd_interval_upsampled = jnp.linspace(
        global_psd_interval[0], global_psd_interval[-1],
        len(global_psd_interval) * upsample_factor
    )

    psd1_upsampled = jnp.interp(global_psd_interval_upsampled, global_psd_interval, psd1)
    psd1_smoothed = gaussian_filter(psd1_upsampled, sigma=smoothing_sigma)

    psd2_upsampled = jnp.interp(global_psd_interval_upsampled, global_psd_interval, psd2)
    psd2_smoothed = gaussian_filter(psd2_upsampled, sigma=smoothing_sigma)


    # Subplot 211: Interval 1
    axs[0].plot(global_psd_interval, label_psd, label='Target (Label_pre)', linestyle='--', color='black')
    axs[0].plot(global_psd_interval_upsampled, psd1_smoothed, label=f'Prediction ({interval1}ms) Smoothed', color='red')
    axs[0].set_title(f"Interval {interval1}ms - Total Loss: {total_loss1:.4f}")
    axs[0].set_xlabel("Frequency (Hz)")
    axs[0].set_ylabel("Normalized Power")
    axs[0].legend()

    stats_text1 = (f"Total Loss: {total_loss1:.4f}\n"
                   f"PSD Loss: {psd_loss1*psd_weight:.4f}\n"
                   f"FR Penalty: {firing_rate_penalty1*firing_rate_weight:.4f}\n"
                   f"Mean FR: {mean_fr1:.2f} Hz")
    axs[0].text(0.05, 0.95, stats_text1,
                transform=axs[0].transAxes, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

    # Subplot 212: Interval 2
    axs[1].plot(global_psd_interval, label_psd_2, label='Target (Label_stim)', linestyle='--', color='black')
    axs[1].plot(global_psd_interval_upsampled, psd2_smoothed, label=f'Prediction ({interval2}ms) Smoothed', color='blue')
    axs[1].set_title(f"Interval {interval2}ms")
    axs[1].set_xlabel("Frequency (Hz)")
    axs[1].set_ylabel("Normalized Power")
    axs[1].legend()
    axs[1].text(0.05, 0.95, f"Mean FR: {mean_fr2:.2f} Hz",
                transform=axs[1].transAxes, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

    plt.tight_layout()
    if savename:
        plt.savefig(savename, format='svg')
    plt.show()

    return total_loss1

def test_net_vis_comp_tfr(net, params, input_scalar, label_psd=None, interval1=(1, 500), interval2=(501, 1000), savename=None):
    """
    Visualizes the model raster and spectrogram.
    """
    # 1. Simulate
    traces = simulate(params, input_scalar)

    # 2. Raster
    spike_threshold = -20.0
    num_neurons = traces.shape[0]

    downsampling_factor = int(jnp.ceil(1.0 / dt_global))
    if downsampling_factor <= 0:
        downsampling_factor = 1

    neuron_traces_data = traces[:, 1:] # Shape (num_neurons, original_num_timepoints)
    original_num_timepoints = neuron_traces_data.shape[1]

    full_res_spike_trains = jnp.zeros((num_neurons, original_num_timepoints), dtype=jnp.float32)
    for i in range(num_neurons):
        neuron_trace = traces[i, 1:]
        # Fix: Changed neuron_trace[:, 1:] to neuron_trace[1:] for 1D array
        spikes_detected = (neuron_trace[:-1] < spike_threshold) & (neuron_trace[1:] >= spike_threshold)
        spike_indices = jnp.where(spikes_detected)[0] + 1
        full_res_spike_trains = full_res_spike_trains.at[i, spike_indices].set(1.0)

    num_timepoints_raster = (original_num_timepoints + downsampling_factor - 1) // downsampling_factor
    spike_image = jnp.zeros((num_neurons, num_timepoints_raster), dtype=jnp.float32)

    for i in range(num_neurons):
        neuron_trace = traces[i, 1:]
        neuron_trace_down = neuron_trace[::downsampling_factor]

        spikes = (neuron_trace_down[:-1] < spike_threshold) & (neuron_trace_down[1:] >= spike_threshold)

        spike_indices = jnp.where(spikes)[0]
        spike_image = spike_image.at[i, spike_indices].set(1.0)

    print(spike_image.shape)

    # Cumulative spiking
    cumulative_spiking = jnp.sum(spike_image, axis=0)

    # 3. Spectrogram (Summed across neurons)
    fs = 1000.0 / dt_global
    nperseg = int(200.0 * fs / 1000.0) # 200ms window
    noverlap = int(nperseg * 0.99)

    # Get raw data (exclude V_init)
    traces_data = traces[:, 1:]

    # Extend traces with mirroring
    traces_extended = extend_traces_for_spectrogram(traces_data, nperseg)

    # Compute summed spectrogram
    f, t, Sxx_sum = compute_summed_spectrogram(traces_extended, fs, nperseg, noverlap)

    # Normalize summed Sxx for plotting
    Sxx_plot = Sxx_sum #np.sqrt(Sxx_sum)

    t_shift = (nperseg / 2) / fs
    t_ms = (t - t_shift) * 1000.0

    # 4. Plot
    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(3, 1, height_ratios=[1, 0.1, 1], hspace=0.15)

    axs = []
    axs.append(fig.add_subplot(gs[0]))
    axs.append(fig.add_subplot(gs[1], sharex=axs[0]))
    axs.append(fig.add_subplot(gs[2], sharex=axs[0]))

    # Raster
    t_max = traces.shape[1] * dt_global
    axs[0].imshow(spike_image, aspect='auto', cmap='Greys', origin='lower', interpolation='none',
                  extent=[0, t_max, 0, spike_image.shape[0]])
    axs[0].set_title("Model Raster")
    axs[0].set_ylabel("Neuron Index")
    axs[0].tick_params(labelbottom=False)

    # Cumulative Spiking
    raster_time_axis = jnp.linspace(0, t_max, num_timepoints_raster)
    axs[1].plot(raster_time_axis, cumulative_spiking, color='black', linewidth=0.8)
    axs[1].set_ylabel("Spike Count")
    axs[1].tick_params(labelbottom=False)

    # Spectrogram
    # Limit freq axis to interest (e.g. 0-100Hz)
    f_lim_idx = f <= 100.0
    f_plot = f[f_lim_idx]
    Sxx_plot_lim = Sxx_plot[f_lim_idx, :]

    # Spectrogram
    vis_smoothed_spectrogram(axs[2], Sxx_plot_lim, [t_ms[0], t_ms[-1]], [f_plot[0], f_plot[-1]])

    axs[2].set_title("Summed Spectrogram (Population Power)")
    axs[2].set_ylabel("Frequency (Hz)")
    axs[2].set_xlabel("Time (ms)")
    axs[2].set_ylim(0, 100)

    # Add red dashed lines
    red_line_freqs = [3.0, 8.0, 12.0, 32.0, 90.0]
    for freq in red_line_freqs:
        axs[2].axhline(y=freq, color='red', linestyle='--', linewidth=1.0, alpha=0.8)

    plt.tight_layout()
    if savename:
        plt.savefig(savename, format='svg')
    plt.show()


def trace_vis_tfr(traces, label_psd=None, interval1=(1, 500), interval2=(501, 1000), dt_trace=0.1):
    """
    Visualizes the model raster and spectrogram.
    """
    # 2. Raster
    spike_threshold = -20.0
    num_neurons = traces.shape[0]

    downsampling_factor = int(jnp.ceil(1.0 / dt_global))
    if downsampling_factor <= 0:
        downsampling_factor = 1

    neuron_traces_data = traces[:, 1:] # Shape (num_neurons, original_num_timepoints)
    original_num_timepoints = neuron_traces_data.shape[1]

    full_res_spike_trains = jnp.zeros((num_neurons, original_num_timepoints), dtype=jnp.float32)
    for i in range(num_neurons):
        neuron_trace = traces[i, 1:]
        # Fix: Changed neuron_trace[:, 1:] to neuron_trace[1:] for 1D array
        spikes_detected = (neuron_trace[:-1] < spike_threshold) & (neuron_trace[1:] >= spike_threshold)
        spike_indices = jnp.where(spikes_detected)[0] + 1
        full_res_spike_trains = full_res_spike_trains.at[i, spike_indices].set(1.0)

    num_timepoints_raster = (original_num_timepoints + downsampling_factor - 1) // downsampling_factor
    spike_image = jnp.zeros((num_neurons, num_timepoints_raster), dtype=jnp.float32)

    for i in range(num_neurons):
        neuron_trace = traces[i, 1:]
        neuron_trace_down = neuron_trace[::downsampling_factor]

        spikes = (neuron_trace_down[:-1] < spike_threshold) & (neuron_trace_down[1:] >= spike_threshold)

        spike_indices = jnp.where(spikes)[0]
        spike_image = spike_image.at[i, spike_indices].set(1.0)

    print(spike_image.shape)

    # Cumulative spiking
    cumulative_spiking = jnp.sum(spike_image, axis=0)

    # 3. Spectrogram (Summed across neurons)
    fs = 1000.0 / dt_global
    nperseg = int(200.0 * fs / 1000.0) # 200ms window
    noverlap = int(nperseg * 0.99)

    # Get raw data (exclude V_init)
    traces_data = traces[:, 1:]

    # Extend traces with mirroring
    traces_extended = extend_traces_for_spectrogram(traces_data, nperseg)

    # Compute summed spectrogram
    f, t, Sxx_sum = compute_summed_spectrogram(traces_extended, fs, nperseg, noverlap)

    # Normalize summed Sxx for plotting
    Sxx_plot = Sxx_sum #np.sqrt(Sxx_sum)

    t_shift = (nperseg / 2) / fs
    t_ms = (t - t_shift) * 1000.0

    # 4. Plot
    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(3, 1, height_ratios=[1, 0.1, 1], hspace=0.15)

    axs = []
    axs.append(fig.add_subplot(gs[0]))
    axs.append(fig.add_subplot(gs[1], sharex=axs[0]))
    axs.append(fig.add_subplot(gs[2], sharex=axs[0]))

    # Raster
    t_max = traces.shape[1] * dt_global
    axs[0].imshow(spike_image, aspect='auto', cmap='Greys', origin='lower', interpolation='none',
                  extent=[0, t_max, 0, spike_image.shape[0]])
    axs[0].set_title("Model Raster")
    axs[0].set_ylabel("Neuron Index")
    axs[0].tick_params(labelbottom=False)

    # Cumulative Spiking
    raster_time_axis = jnp.linspace(0, t_max, num_timepoints_raster)
    axs[1].plot(raster_time_axis, cumulative_spiking, color='black', linewidth=0.8)
    axs[1].set_ylabel("Spike Count")
    axs[1].tick_params(labelbottom=False)

    # Spectrogram
    # Limit freq axis to interest (e.g. 0-100Hz)
    f_lim_idx = f <= 100.0
    f_plot = f[f_lim_idx]
    Sxx_plot_lim = Sxx_plot[f_lim_idx, :]

    # Spectrogram
    vis_smoothed_spectrogram(axs[2], Sxx_plot_lim, [t_ms[0], t_ms[-1]], [f_plot[0], f_plot[-1]])

    axs[2].set_title("Summed Spectrogram (Population Power)")
    axs[2].set_ylabel("Frequency (Hz)")
    axs[2].set_xlabel("Time (ms)")
    axs[2].set_ylim(0, 100)

    # Add red dashed lines
    red_line_freqs = [3.0, 8.0, 12.0, 32.0, 90.0]
    for freq in red_line_freqs:
        axs[2].axhline(y=freq, color='red', linestyle='--', linewidth=1.0, alpha=0.8)

    plt.tight_layout()
    if savename:
        plt.savefig(savename, format='svg')
    plt.show()


def save_jnn(filename, filepath, net_object, initial_params, mid_params, final_params, log_net):
    """
    Saves various components of the JAXley model to a single .pkl file using pickle.
    Args:
        filename (str):
        filepath (str): Directory path to save the file.
        net_object (jx.Network): The JAXley network object.
        initial_params: Initial parameters of the network.
        mid_params: Intermediate parameters (if any).
        final_params: Final trained parameters of the network.
        log_net: Training log data.
    """
    full_path = os.path.join(filepath, filename + ".pkl") # Changed extension to .pkl

    # Collect all data into a dictionary
    model_data = {
        'initial_params': initial_params,
        'mid_params': mid_params,
        'final_params': final_params,
        'log_net': log_net,
        'net_params': net_object.get_parameters(), # Save the current parameters of the net
        'Ne': Ne, # Using global Ne variable
        'Nig': Nig, # Using global Nig variable
        'Nil': Nil, # Using global Nil variable
    }

    with open(full_path, "wb") as handle:
        pickle.dump(model_data, handle) # Use pickle.dump directly
    print(f"Model components saved to {full_path}")


def load_jnn(filename, filepath):
    """
    Loads various components of the JAXley model from a .pkl file using pickle.
    Args:
        filename (str): Base name for the file (e.g., "model_001").
        filepath (str): Directory path to load the file from.
    Returns:
        tuple: (rebuilt_net_object, initial_params, mid_params, final_params, log_net)
              where rebuilt_net_object is recreated and its parameters set.
    """
    full_path = os.path.join(filepath, filename) # .pkl
    with open(full_path, "rb") as handle:
        loaded_data = pickle.load(handle) # Use pickle.load directly

    initial_params = loaded_data['initial_params']
    mid_params = loaded_data['mid_params']
    final_params = loaded_data['final_params']
    log_net = loaded_data['log_net']
    net_params = loaded_data['net_params']

    rebuilt_net = net_eig(loaded_data['Ne'], loaded_data['Nig'], loaded_data['Nil'])
    print(f"Model components loaded from {full_path}")
    return rebuilt_net, initial_params, mid_params, final_params, log_net


class GradedGABAa(Synapse):
    """
    Graded fast Inhibitory Synapse. (GABAa)
    Ref: Golowasch et al. (1999) ; Prinz, A. A., et al. (2004).
    """
    def __init__(self, tauD_GABAa: Optional[float] = None):
        super().__init__()

        r_d_gSyn = np.random.uniform(4.0, 6.0)

        self.synapse_params = {
            "gGABAa": r_d_gSyn,       # Reduced initial conductance
            "EGABAa": -80.0,     # Reversal Potential (mV)
            "tauDGABAa": 5.0,        # Decay (ms)
            "tauRGABAa": 0.5,         # Rise (ms)
            "slopeGABAa": 5.0,        # Steepness of activation
            "V_thGABAa": -20.0        # Threshold (mV) - Tuned to activate during spikes
        }

        if tauD_GABAa is not None:
            self.synapse_params["tauDGABAa"] = tauD_GABAa

        self.synapse_states = {"sGABAa": 0.1}

    def update_states(self, states, dt, pre_v, post_v, params):
        # s' = -s/tauD + 0.5*(1+tanh((V-Vth)/slope)) * ((1-s)/tauR)
        s = states["sGABAa"]
        activation = 0.5 * (1 + jnp.tanh((pre_v - params["V_thGABAa"]) / params["slopeGABAa"]))
        d_s = (-s / params["tauDGABAa"]) + activation * ((1 - s) / params["tauRGABAa"])
        return {"sGABAa": s + d_s * dt}

    def compute_current(self, states, pre_v, post_v, params):
        return params["gGABAa"] * states["sGABAa"] * (post_v - params["EGABAa"])


class GradedGABab(Synapse):
    """
    Graded slow Inhibitory Synapse. (GABab)
    Ref: Golowasch et al. (1999) ; Prinz, A. A., et al. (2004).
    """
    def __init__(self, tauD_GABab: Optional[float] = None):
        super().__init__()
        r_d_gSyn = np.random.uniform(0.5, 9.5)
        self.synapse_params = {
            "gGABab": r_d_gSyn,       # Conductance (uS)
            "EGABab": -95.0,     # Reversal Potential (mV)
            "tauDGABab": 200.0,        # Decay (ms)
            "tauRGABab": 10.0,         # Rise (ms)
            "slopeGABab": 5.0,        # Steepness of activation
            "V_thGABab": -20.0        # Threshold (mV) - Tuned to activate during spikes
        }

        if tauD_GABab is not None:
            self.synapse_params["tauDGABab"] = tauD_GABab

        self.synapse_states = {"sGABab": 0.01}

    def update_states(self, states, dt, pre_v, post_v, params):
        # s' = -s/tauD + 0.5*(1+tanh((V-Vth)/slope)) * ((1-s)/tauR)
        s = states["sGABab"]
        activation = 0.5 * (1 + jnp.tanh((pre_v - params["V_thGABab"]) / params["slopeGABab"]))
        d_s = (-s / params["tauDGABab"]) + activation * ((1 - s) / params["tauRGABab"])
        return {"sGABab": s + d_s * dt}

    def compute_current(self, states, pre_v, post_v, params):
        return params["gGABab"] * states["sGABab"] * (post_v - params["EGABab"])


class GradedAMPA(Synapse):
    """
    Graded Excitatory Synapse.
    Same dynamics as GradedGABAa, but with E_rev = 0 mV and faster kinetics
    """
    def __init__(self, tauD_AMPA: Optional[float] = None):
        super().__init__()
        r_d_gSyn = np.random.uniform(2.0, 3.0)
        self.synapse_params = {
            "gAMPA": r_d_gSyn,       # Reduced initial conductance
            "EAMPA": 0.0,        # Reversal Potential (mV) - EXCITATORY
            "tauDAMPA": 5.0,         # Faster decay for AMPA
            "tauRAMPA": 0.2,         # Fast rise
            "slopeAMPA": 5.0,
            "V_thAMPA": -20.0        # Threshold (mV)
        }

        if tauD_AMPA is not None:
            self.synapse_params["tauDAMPA"] = tauD_AMPA

        self.synapse_states = {"sAMPA": 0.1}

    def update_states(self, states, dt, pre_v, post_v, params):
        s = states["sAMPA"]
        # Using same naming convention for internal params, mapped to AMPA
        activation = 0.5 * (1 + jnp.tanh((pre_v - params["V_thAMPA"]) / params["slopeAMPA"]))
        d_s = (-s / params["tauDAMPA"]) + activation * ((1 - s) / params["tauRAMPA"])
        return {"sAMPA": s + d_s * dt}

    def compute_current(self, states, pre_v, post_v, params):
        return params["gAMPA"] * states["sAMPA"] * (post_v - params["EAMPA"])


class Inoise(Channel):
    """
    Stochastic Ornstein-Uhlenbeck noise channel.
    """
    def __init__(self, name: str = None, initial_seed: Optional[int] = None, initial_amp_noise: Optional[float] = None, initial_tau: Optional[float] = None, initial_mean: Optional[float] = None):
        self.current_is_in_mA_per_cm2 = True
        super().__init__(name)

        self.channel_params = {
            "amp_noise": 0.01,  # Reduced initial noise amplitude
            "mean": 0.00,        # The baseline current drive [mA/cm^2]
            "tau": 20.0         # Correlation time constant [ms]
        }
        # If no initial_seed is provided, generate a random one using numpy.
        # This ensures each Inoise instance gets a unique starting seed.
        if initial_seed is None:
            self.channel_params["seed"] = float(np.random.randint(0, 2**16 - 1))
        else:
            self.channel_params["seed"] = float(initial_seed)

        if initial_amp_noise is None:
            self.channel_params["amp_noise"] = float(0.01) # Ensure this default is also reduced
        else:
            self.channel_params["amp_noise"] = float(initial_amp_noise)

        if initial_tau is None:
            self.channel_params["tau"] = float(20.0)
        else:
            self.channel_params["tau"] = float(initial_tau)

        if initial_mean is None:
            self.channel_params["mean"] = float(0.00)
        else:
            self.channel_params["mean"] = float(initial_mean)

        self.channel_states = {"n": 0.00, "step": 0.0}
        self.current_name = "i_noise"

    def update_states(self, states, dt, v, params):
        """
        Updates the noise state 'n' using an Ornstein-Uhlenbeck process.
        """
        n = states["n"]
        step = states["step"]

        # 1. RNG Handling
        # When JAXley vmaps the update_states function, params["seed"] will be an array.
        # All other inputs (n, step, v, dt) will also be batched (arrays).
        # We need to vmap the PRNGKey and fold_in calls.

        # Ensure seed is an integer type for PRNGKey
        seeds_int = params["seed"].astype(int)

        # Create base keys (potentially batched)
        if seeds_int.ndim == 0:
            base_key = jax.random.PRNGKey(seeds_int)
        else:
            base_key = jax.vmap(jax.random.PRNGKey)(seeds_int)

        # Fold in step (potentially batched)
        if step.ndim == 0:
            step_key = jax.random.fold_in(base_key, step.astype(int))
        else: # if step is an array
            step_key = jax.vmap(jax.random.fold_in)(base_key, step.astype(int))

        # Generate normal random numbers (potentially batched)
        # A single JAX PRNGKey has shape (2,) and ndim=1.
        # An array of N JAX PRNGKeys has shape (N, 2) and ndim=2.
        if step_key.ndim == 1: # if step_key is a single key
            xi = jax.random.normal(step_key)
        else: # if step_key is an array of keys (ndim 2 for key array)
            xi = jax.vmap(jax.random.normal)(step_key)

        # 2. Physics (Ornstein-Uhlenbeck)
        # dn = -(n - mean)/tau * dt + sigma*sqrt(2/tau)*dW
        mu = params["mean"]
        sigma = params["amp_noise"]
        tau = params["tau"]

        drift = (mu - n) / tau * dt
        diffusion = sigma * jnp.sqrt(2.0 / tau) * xi * jnp.sqrt(dt_global) # FIX: Changed dt to dt_global

        new_n = n + drift + diffusion # FIX: Changed 'n' to 'n_prev'

        # Return new state AND increment step
        return {"n": new_n, "step": step + 1.0}

    def compute_current(self, states, v, params):
        """
        Returns the current.
        Note: We return negative 'n' so that a positive mean acts
        as an excitatory (depolarizing) injection in the cable equation.
        """
        return -states["n"]

    def init_state(self, states, v, params, delta_t):
        """
        Initialize to the mean value.
        This needs to handle batched parameters if JAXley vmaps init_state.
        """
        # If params["mean"] is a scalar, jnp.zeros_like(params["mean"]) is a scalar 0.0.
        # If params["mean"] is an array, jnp.zeros_like(params["mean"]) is an array of 0.0s.
        # This handles both cases correctly.
        return {"n": jnp.zeros_like(params["mean"]) + params["mean"], "step": jnp.zeros_like(params["mean"]) + 0.0}


class Dataset:
    """
    A simple Dataloader which returns batches of the data.

    Instead of using this simple dataloader, you can also just use one from
    PyTorch or Tensorflow. You do not have to understand what is going on here
    to follow this tutorial.
    """

    def __init__(self, inputs: np.ndarray, labels: np.ndarray):
        """
        Initialize the dataloader.

        Args:
            inputs: Array of shape (num_samples, num_dim)
            labels: Array of shape (num_samples,)
        """
        assert len(inputs) == len(labels), "Inputs and labels must have same length"
        self.inputs = inputs
        self.labels = labels
        self.num_samples = len(inputs)
        self._rng_state = None
        self.batch_size = 1

    def shuffle(self, seed=None):
        """
        Shuffle the dataset in-place
        """
        self._rng_state = np.random.get_state()[1][0] if seed is None else seed
        np.random.seed(self._rng_state)
        indices = np.random.permutation(self.num_samples)
        self.inputs = self.inputs[indices]
        self.labels = self.labels[indices]
        return self

    def batch(self, batch_size):
        """
        Create batches of the data.
        """
        self.batch_size = batch_size
        return self

    def __iter__(self):
        """
        Iterate over the dataset.
        """
        self.shuffle(seed=self._rng_state)
        for start in range(0, self.num_samples, self.batch_size):
            end = min(start + self.batch_size, self.num_samples)
            yield self.inputs[start:end], self.labels[start:end]
        self._rng_state += 1


class ClampTransform(jt.Transform):
    def __init__(self, lower_bound, upper_bound):
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound

    def forward(self, x):
        return jnp.nan_to_num(jnp.clip(x, self.lower_bound, self.upper_bound), 0.0)

    def inverse(self, x):
        return x


@dataclass
class SDRState:
    momentum_accum: Any
    step_count: int

def SDR(
    learning_rate: float = 1e-3,
    momentum: float = 0.1,
    change_lower_bound: float = -1.0,
    change_upper_bound: float = 1.0,
    delta_distribution: Callable = jax.random.uniform,
    sigma: float = 1.0
) -> optax.GradientTransformation:
    """
    Optax-compliant implementation of the Stochastic Delta Rule (SDR).
    Includes x32 stability transform (Clamp/OmitNaN).
    """
    def init_fn(params):
        momentum_accum = jax.tree.map(lambda p: jnp.zeros_like(p), params)
        return SDRState(
            momentum_accum=momentum_accum,
            step_count=0
        )

    def update_fn(updates, state, params=None, value=None, key=None):
        if key is None:
            raise ValueError("SDR requires a random 'key' to be passed to update().")

        grads = updates

        new_momentum_accum = jax.tree.map(
            lambda m, g: momentum * m + g,
            state.momentum_accum, grads
        )

        grad_signs = jax.tree.map(jnp.sign, new_momentum_accum)

        param_leaves, treedef = jax.tree.flatten(grads)
        subkeys = jax.random.split(key, len(param_leaves))
        param_keys_tree = jax.tree.unflatten(treedef, subkeys)

        random_factors = jax.tree.map(
            lambda g, k: sigma * delta_distribution(k, g.shape),
            grads, param_keys_tree
        )

        def smooth_factor(x):
            if x.ndim == 2:
                n, m = x.shape
                kn = int(np.sqrt(n))
                km = int(np.sqrt(m))
                kn = max(1, kn)
                km = max(1, km)
                kernel = jnp.ones((kn, km)) / (kn * km)
                return jax.scipy.signal.convolve2d(x, kernel, mode='same')
            elif x.ndim == 1:
                n = x.shape[0]
                k = int(np.sqrt(n))
                k = max(1, k)
                kernel = jnp.ones((k,)) / k
                return jax.scipy.signal.convolve(x, kernel, mode='same')
            else:
                return x

        random_factors = jax.tree.map(smooth_factor, random_factors)

        raw_updates = jax.tree.map(
            lambda s, r: -learning_rate * s * r,
            grad_signs, random_factors
        )

        boundTransform = ClampTransform(change_lower_bound, change_upper_bound)
        final_updates = jax.tree.map(lambda x: boundTransform.forward(x), raw_updates)

        new_state = SDRState(
            momentum_accum=new_momentum_accum,
            step_count=state.step_count + 1
        )

        return final_updates, new_state

    return optax.GradientTransformation(init_fn, update_fn)


@dataclass
class GSDRState:
    inner_state: Any
    params_opt: Any
    inner_state_opt: Any
    loss_opt: float
    a: float
    a_opt: float
    lambda_d: float
    step_count: int
    consecutive_unchanged_epochs: int
    last_optimal_change_step: int

def GSDR(
    inner_optimizer: optax.GradientTransformation,
    delta_distribution: Callable = jax.random.normal,
    deselection_threshold: float = 2.0,
    a_init: float = 0.5,
    lambda_d: float = 1.0,
    checkpoint_n: int = 10,
    tau_a_growth: float = 10.0,
    mcdp: bool = True,
    a_dynamic: bool = False
) -> optax.GradientTransformation:
    """
    Optax-compliant implementation of the Genetic-Stochastic Delta Rule.
    """

    def init_fn(params):
        inner_state = inner_optimizer.init(params)
        return GSDRState(
            inner_state=inner_state,
            params_opt=params,
            inner_state_opt=inner_state,
            loss_opt=jnp.inf,
            a=a_init,
            a_opt=a_init,
            lambda_d=lambda_d,
            step_count=0,
            consecutive_unchanged_epochs=0,
            last_optimal_change_step=0
        )

    def update_fn(updates, state, params=None, value=None, key=None, mcdp_factors=None):

        if params is None:
            raise ValueError("GSDR requires 'params' to be passed to update().")
        if value is None:
            raise ValueError("GSDR requires current loss 'value' to be passed to update().")
        if key is None:
            raise ValueError("GSDR requires a random 'key' to be passed to update().")

        grads = updates
        loss = value

        # 1. Update Best-Known State (Optimality Check)
        is_new_opt = (loss < state.loss_opt)

        new_params_opt = jax.tree.map(
            lambda cur, opt: jnp.where(is_new_opt, cur, opt),
            params, state.params_opt
        )
        new_loss_opt = jnp.where(is_new_opt, loss, state.loss_opt)
        new_a_opt = jnp.where(is_new_opt, state.a, state.a_opt)
        new_inner_state_opt = jax.tree.map(
            lambda cur, opt: jnp.where(is_new_opt, cur, opt),
            state.inner_state, state.inner_state_opt
        )

        next_consecutive_unchanged_epochs = jnp.where(
            is_new_opt, 0, state.consecutive_unchanged_epochs + 1
        )
        step_of_last_optimal_change = jnp.where(
            is_new_opt, state.step_count + 1, state.last_optimal_change_step
        )

        # 2. Check Reset Conditions
        is_deselect = ((loss > (new_loss_opt * deselection_threshold)) & (new_loss_opt != jnp.inf)) | (jnp.isnan(loss))
        is_reset_due_to_checkpoint = (state.step_count > 0) & \
                                     (next_consecutive_unchanged_epochs >= checkpoint_n) & \
                                     (new_loss_opt != jnp.inf)

        should_reset = is_deselect | is_reset_due_to_checkpoint

        def reset_branch(operand):
            _params, _new_params_opt, _new_inner_state_opt, _current_step = operand

            reset_updates = jax.tree.map(
                lambda opt_p, cur_p: opt_p - cur_p,
                _new_params_opt, _params
            )

            reset_state = GSDRState(
                inner_state=_new_inner_state_opt,
                params_opt=_new_params_opt,
                inner_state_opt=_new_inner_state_opt,
                loss_opt=new_loss_opt,
                a=new_a_opt,
                a_opt=new_a_opt,
                lambda_d=state.lambda_d,
                step_count=_current_step,
                consecutive_unchanged_epochs=0,
                last_optimal_change_step=_current_step
            )

            if is_reset_due_to_checkpoint:
                print(">Deselection(Checkpoint)")
            elif is_deselect:
                print(">Deselection(Divergence)")

            return reset_updates, reset_state

        def normal_branch(operand):
            _params, _new_params_opt, _new_inner_state_opt, _current_step = operand

            time_since_last_change = jnp.maximum(0, _current_step - step_of_last_optimal_change)
            effective_lambda_d = (time_since_last_change**2) * (1.0 - jnp.exp(-(time_since_last_change) / tau_a_growth))

            inner_opt_key, delta_dist_key, a_key, noise_key = jax.random.split(key, 4)

            if a_dynamic:
                delta_a = jax.random.uniform(a_key, minval=-.1, maxval=.1)
                a_candidate = state.a + delta_a
                next_a = jax.numpy.clip(a_candidate, 0.0, 1.0)
            else:
                next_a = state.a

            boundTransform = ClampTransform(-1.0, 1.0)

            inner_updates, updated_inner_state = inner_optimizer.update(grads, state.inner_state, _params, key=inner_opt_key)
            inner_updates = jax.tree.map(lambda x: boundTransform.forward(x), inner_updates)

            # Fixed typo here: jax.tree_util instead of jax.tree.util
            inner_updates_flat = jax.tree_util.tree_leaves(inner_updates)
            avg_inner_update = jnp.mean(jnp.concatenate([x.flatten() for x in inner_updates_flat])) if inner_updates_flat else jnp.array(0.0)
            std_inner_update = jnp.std(jnp.concatenate([x.flatten() for x in inner_updates_flat])) if inner_updates_flat else jnp.array(0.0)
            jax.debug.print("GSDR: Avg Inner Update: {}, Std Inner Update: {}", avg_inner_update, std_inner_update)

            param_leaves, treedef = jax.tree.flatten(_params)
            subkeys = jax.random.split(noise_key, len(param_leaves))
            param_keys_tree = jax.tree.unflatten(treedef, subkeys)

            delta_d = jax.tree.map(
                lambda p, k: delta_distribution(k, p.shape),
                _params, param_keys_tree
            )

            if mcdp and mcdp_factors is not None:
                delta = jax.tree.map(lambda n, p, r: n * p * r, delta_d, _params, mcdp_factors)
            else:
                delta = jax.tree.map(lambda n, p: n * p, delta_d, _params)

            delta = jax.tree.map(lambda x: boundTransform.forward(x), delta)

            # Fixed typo here
            delta_flat = jax.tree_util.tree_leaves(delta)
            avg_delta = jnp.mean(jnp.concatenate([x.flatten() for x in delta_flat])) if delta_flat else jnp.array(0.0)
            std_delta = jnp.std(jnp.concatenate([x.flatten() for x in delta_flat])) if delta_flat else jnp.array(0.0)
            jax.debug.print("GSDR: Avg Delta (MC): {}, Std Delta (MC): {}", avg_delta, std_delta)

            combined_updates = jax.tree.map(
                lambda d, g: effective_lambda_d * (next_a * d + (1 - next_a) * g),
                delta, inner_updates
            )

            # Fixed typo here
            combined_updates_flat = jax.tree_util.tree_leaves(combined_updates)
            avg_combined_update = jnp.mean(jnp.concatenate([x.flatten() for x in combined_updates_flat])) if combined_updates_flat else jnp.array(0.0)
            std_combined_update = jnp.std(jnp.concatenate([x.flatten() for x in combined_updates_flat])) if combined_updates_flat else jnp.array(0.0)
            jax.debug.print("GSDR: Avg Combined Update: {}, Std Combined Update: {}", avg_combined_update, std_combined_update)

            new_normal_state = GSDRState(
                inner_state=updated_inner_state,
                params_opt=_new_params_opt,
                inner_state_opt=_new_inner_state_opt,
                loss_opt=new_loss_opt,
                a=next_a,
                a_opt=new_a_opt,
                lambda_d=state.lambda_d,
                step_count=_current_step,
                consecutive_unchanged_epochs=next_consecutive_unchanged_epochs,
                last_optimal_change_step=step_of_last_optimal_change
            )

            return combined_updates, new_normal_state

        current_step = state.step_count + 1
        operand = (params, new_params_opt, new_inner_state_opt, current_step)

        final_updates, new_state = jax.lax.cond(
            should_reset,
            reset_branch,
            normal_branch,
            operand
        )

        return final_updates, new_state

    return optax.GradientTransformation(init_fn, update_fn)


def noise_current(
    i_delay: float,
    i_dur: float,
    i_amp: float,
    delta_t: float,
    t_max: float,
    seed: int = 0, noise_standard_deviation: Optional[float] = None, noise_correlation_tau: Optional[float] = None, noise_mean: Optional[float] = None) -> jnp.ndarray:
    """
    Generates a random noise current pulse using an Ornstein-Uhlenbeck process.

    Args:
        i_delay: Start time of the current pulse in ms.
        i_dur: Duration of the current pulse in ms.
        i_amp: Overall amplitude scaling of the noise pulse (nA).
        delta_t: Time step for the simulation in ms.
        t_max: Total simulation time in ms.
        seed: Random seed for reproducibility (default: 0).

    Returns:
        jnp.ndarray: Array representing the noise current pulse over time, with shape (int(t_max / delta_t) + 1,).R
    """
    # Parameters for the Ornstein-Uhlenbeck noise process
    if noise_standard_deviation is None:
        noise_standard_deviation = 0.1 # Sigma for OU (nA) - a default value
    if noise_correlation_tau is None:
        noise_correlation_tau = 10.0     # Correlation time constant in ms
    if noise_mean is None:
        noise_mean = 0.1                 # Mean of the OU process

    key = jax.random.PRNGKey(seed)

    # Calculate number of steps to include initial state (for JAXley compatibility)
    num_steps = int(t_max / delta_t) + 1
    time_axis_array = jnp.arange(0, t_max + delta_t, delta_t)

    def ou_step_fn(n_prev, key_t):
        key_noise, _ = jax.random.split(key_t)
        xi = jax.random.normal(key_noise)

        drift = (noise_mean - n_prev) / noise_correlation_tau * delta_t
        diffusion = noise_standard_deviation * jnp.sqrt(2.0 / noise_correlation_tau) * xi * jnp.sqrt(dt_global) # FIX: Changed dt to dt_global

        new_n = n_prev + drift + diffusion # FIX: Changed 'n' to 'n_prev'

        # Return new state AND increment step
        return new_n, new_n

    initial_state_ou = noise_mean
    # Split keys for num_steps-1 iterations, as initial_state is already handled
    keys_series = jax.random.split(key, num_steps - 1)

    # Run the scan to generate the raw noise trace
    _, full_noise_trace_raw = jax.lax.scan(ou_step_fn, initial_state_ou, keys_series)
    # Prepend the initial state to get the full trace length
    full_noise_trace = jnp.concatenate([jnp.array([initial_state_ou]), full_noise_trace_raw])

    # Apply the pulse window based on i_delay and i_dur
    time_off = i_delay + i_dur
    pulse_mask = (time_axis_array >= i_delay) & (time_axis_array <= time_off)

    # Scale the noise by i_amp and apply the pulse mask
    noise_current_pulse_array = jnp.where(pulse_mask, full_noise_trace * i_amp, 0.0)

    return noise_current_pulse_array


def ramp_current(
    i_delay: float,
    i_dur: float,
    i_amp: float,
    delta_t: float,
    t_max: float
) -> jnp.ndarray:
    """
    Generates a ramping current pulse.

    The current is 0 up to i_delay, then ramps up linearly to i_amp over i_dur,
    and then is 0 again after i_delay + i_dur.

    Args:
        i_delay: Start time of the current ramp in ms.
        i_dur: Duration of the current ramp in ms.
        i_amp: Peak amplitude of the ramped current (nA).
        delta_t: Time step for the simulation in ms.
        t_max: Total simulation time in ms.

    Returns:
        jnp.ndarray: Array representing the ramping current pulse over time.
    """
    num_steps = int(t_max / delta_t) + 1
    time_axis_array = jnp.arange(0, t_max + delta_t, delta_t)

    current = jnp.zeros_like(time_axis_array)

    # Define the ramp start and end times
    t_ramp_start = i_delay
    t_ramp_end = i_delay + i_dur

    # Create a mask for the ramping period
    ramp_mask = (time_axis_array >= t_ramp_start) & (time_axis_array <= t_ramp_end)

    # Calculate the linear ramp within the masked period
    # (time - t_ramp_start) / i_dur gives a value from 0 to 1 over the duration
    ramp_factor = (time_axis_array - t_ramp_start) / i_dur
    ramp_current_segment = i_amp * ramp_factor

    # Apply the ramp current only within the masked period
    current = jnp.where(ramp_mask, ramp_current_segment, current)

    return current

def step_current(
    i_delay: float,
    i_dur: float,
    i_amp: float,
    delta_t: float,
    t_max: float
) -> jnp.ndarray:
    """
    Generates a step current pulse.

    The current is 0 up to i_delay, then steps up to i_amp for i_dur,
    and then is 0 again after i_delay + i_dur.

    Args:
        i_delay: Start time of the current step in ms.
        i_dur: Duration of the current step in ms.
        i_amp: Amplitude of the step current (nA).
        delta_t: Time step for the simulation in ms.
        t_max: Total simulation time in ms.

    Returns:
        jnp.ndarray: Array representing the step current pulse over time.
    """
    num_steps = int(t_max / delta_t) + 1
    time_axis_array = jnp.arange(0, t_max + delta_t, delta_t)

    current = jnp.zeros_like(time_axis_array)

    # Define the step start and end times
    t_step_start = i_delay
    t_step_end = i_delay + i_dur

    # Create a mask for the step period
    step_mask = (time_axis_array >= t_step_start) & (time_axis_array <= t_step_end)

    # Apply the step current only within the masked period
    current = jnp.where(step_mask, i_amp, current)

    return current


def noise_current_ac(
    i_delay: float,
    i_dur: float,
    amp_n: float,
    amp_b: float,
    spect: jnp.ndarray,
    delta_t: float,
    t_max: float,
    seed: int = 0
) -> jnp.ndarray:
    """
    Generates a current composed of an Ornstein-Uhlenbeck noise pulse and
    a superposition of sinusoidal waves (AC).

    Formula: I_total = (OU_Noise * amp_n) + (Sum(Sin(spect)) * amp_b)

    Args:
        i_delay: Start time of the pulse window in ms.
        i_dur: Duration of the pulse window in ms.
        amp_n: Amplitude scaling for the OU noise component (nA).
        amp_b: Amplitude scaling for the AC/Sinusoidal component (nA).
        spect: 1D array of frequencies (Hz) to construct the AC signal.
               Example: jnp.array([10.0, 40.0, 100.0]) for 10, 40, and 100Hz.
        delta_t: Time step for the simulation in ms.
        t_max: Total simulation time in ms.
        seed: Random seed for reproducibility.

    Returns:
        jnp.ndarray: The combined current trace (nA).
    """

    num_steps = int(t_max / delta_t) + 1
    time_axis_array = jnp.arange(0, t_max + delta_t, delta_t)

    noise_standard_deviation = 0.1
    noise_correlation_tau = 10.0
    noise_mean = 0.0 # Often 0 for pure fluctuations, user had 0.2 previously

    key = jax.random.PRNGKey(seed)

    def ou_step_fn(n_prev, key_t):
        key_noise, _ = jax.random.split(key_t)
        xi = jax.random.normal(key_noise)

        # Euler-Maruyama method
        drift = (noise_mean - n_prev) / noise_correlation_tau * delta_t
        # Fixed: replaced 'dt_global' with passed 'delta_t'
        diffusion = noise_standard_deviation * jnp.sqrt(2.0 / noise_correlation_tau) * xi * jnp.sqrt(delta_t)

        new_n = n_prev + drift + diffusion
        return new_n, new_n

    initial_state_ou = noise_mean
    keys_series = jax.random.split(key, num_steps - 1)

    _, full_noise_trace_raw = jax.lax.scan(ou_step_fn, initial_state_ou, keys_series)
    full_noise_trace = jnp.concatenate([jnp.array([initial_state_ou]), full_noise_trace_raw])

    # Conversion: t is in ms, spect is in Hz (1/s).
    # Argument = 2*pi * f * (t / 1000)

    t_reshaped = time_axis_array[None, :]
    freqs_reshaped = spect[:, None]

    # Calculate phases
    phases = 2 * jnp.pi * freqs_reshaped * (t_reshaped / 1000.0)

    # Sum sines across the frequency axis (axis 0)
    # Result is shape (T,)
    ac_trace = jnp.sum(jnp.sin(phases), axis=0)

    # --- 4. Combine and Gate ---
    # Define the time window mask
    time_off = i_delay + i_dur
    pulse_mask = (time_axis_array >= i_delay) & (time_axis_array <= time_off)

    # Apply scaling
    weighted_noise = full_noise_trace * amp_n
    weighted_ac = ac_trace * amp_b

    # Combine
    # Note: Depending on requirements, you might want the AC to run continuously
    # or only during the pulse. Here I apply the mask to BOTH components.
    total_current = jnp.where(pulse_mask, weighted_noise + weighted_ac, 0.0)

    return total_current


def plot_full_simulation_summary(recorded_voltages, time_axis, dt_global,
                                 spike_threshold=-20.0,
                                 window_size=250.0, overlap=0.95,
                                 f_min=1.0, f_max=100.0,
                                 title_suffix="", figsize=(16, 14), save=False, savename="fig3p.svg",
                                 aperiodic_correct: float = 0.0,
                                 baseline_relative: Optional[Tuple[float, float]] = None,
                                 interval1: Tuple[float, float] = (1, 500),
                                 interval2: Tuple[float, float] = (500, 1000)): # Added baseline_relative
    """
    Combines voltage image, raster plot, and time-frequency response into a single 3x1 subplot figure.
    """
    # Replace NaN values with 0 at the very beginning
    recorded_voltages = jnp.nan_to_num(recorded_voltages, nan=0.0)
    recorded_voltages = jnp.clip(recorded_voltages, -100, +100)

    fig = plt.figure(figsize=figsize) # Adjust figure size for 3 tall subplots

    # Subplot 1: Raster Plot as Image
    raster_ax = plt.subplot(4, 1, 1) # Changed to 3, 1, 1
    num_neurons = recorded_voltages.shape[0]

    # Calculate downsampling factor for spike detection prior to thresholding
    # e.g., if dt_global=0.1ms, factor=10, meaning spikes are detected at ~1ms resolution.
    downsampling_factor = int(jnp.ceil(1.0 / dt_global))
    if downsampling_factor <= 0:
        downsampling_factor = 1

    # Get the voltage data, excluding the initial V_init column
    neuron_traces_data = recorded_voltages[:, 1:] # Shape (num_neurons, original_num_timepoints)
    original_num_timepoints = neuron_traces_data.shape[1]

    # Create a binary spike train for each neuron at full resolution
    full_res_spike_trains = jnp.zeros((num_neurons, original_num_timepoints), dtype=jnp.float32)
    for i in range(num_neurons):
        neuron_trace = recorded_voltages[i, 1:] # Exclude V_init
        spikes_detected = (neuron_trace[:-1] < spike_threshold) & (neuron_trace[1:] >= spike_threshold)
        spike_indices = jnp.where(spikes_detected)[0] + 1 # +1 to mark the point after crossing
        full_res_spike_trains = full_res_spike_trains.at[i, spike_indices].set(1.0)


    # The spike_image creation for raster plotting:
    # Calculate the number of time points in the downsampled raster
    num_timepoints_raster = (original_num_timepoints + downsampling_factor - 1) // downsampling_factor

    # Create a 2D array for the downsampled spike image
    spike_image = jnp.zeros((num_neurons, num_timepoints_raster), dtype=jnp.float32)

    for i in range(num_neurons):
        # Extract and downsample the voltage trace for the current neuron
        # We take samples at intervals of 'downsampling_factor'
        neuron_trace = recorded_voltages[i, 1:]
        neuron_trace_down = neuron_trace[::downsampling_factor]

        # Detect spikes on downsampled
        spikes = (neuron_trace_down[:-1] < spike_threshold) & (neuron_trace_down[1:] >= spike_threshold)

        # Map downsampled spike indices to image
        spike_indices = jnp.where(spikes)[0]
        spike_image = spike_image.at[i, spike_indices].set(1.0)


    # The `time_axis` provided is for the original, full-resolution data. When plotting the
    # downsampled `spike_image`, its visual extent should still cover the full original time range.
    # The `extent` parameter of `imshow` handles this scaling automatically.
    plt.imshow(
        spike_image,
        aspect='auto',
        cmap='Greys',
        origin='lower',
        extent=[time_axis[0], time_axis[-1], 0, num_neurons], # Use original time extent for display
        vmin=0, vmax=1
    )
    # Add a custom colorbar to indicate spikes
    cbar = plt.colorbar(label='Spike Activity', ticks=[0.25, 0.75]) # Position ticks to be in the middle of bands
    cbar.set_ticklabels(['No Spike', 'Spike'])

    plt.title(f'Raster Plot (Image) {title_suffix}')
    plt.xlabel('Time (ms)')
    plt.ylabel('Neuron Index')

    # Generate y-ticks at approximately 10% intervals
    if num_neurons > 0:
        y_tick_interval = max(1, int(np.ceil(num_neurons / 10)))
        y_ticks = np.arange(0, num_neurons + 1, y_tick_interval)
        # Ensure 0 is always included and last tick is within bounds or is the last neuron
        y_ticks = np.unique(np.clip(y_ticks, 0, num_neurons - 1)).astype(int)
        plt.yticks(y_ticks)
    else:
        plt.yticks([])

    plt.ylim(-0.5, num_neurons - 0.5)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.xlim([time_axis[0], time_axis[-1]])
    plt.axvline(x=500, color='red', linestyle='--', linewidth=1.5)

    # Subplot 2: Time-Frequency Response
    plt.subplot(4, 1, 2)
    recorded_voltages = jnp.nan_to_num(recorded_voltages, nan=0.0, posinf=0.0, neginf=0.0)

    # --- NEW: Calculate recorded_spiking_w and prepare for spectrogram ---
    w = 5.0 # Gaussian kernel width, as per request default
    # Sampling frequency of the SPIKE IMAGE (1 kHz if downsampling_factor ~ 1ms)
    dt_raster = dt_global * downsampling_factor
    fs_raster = 1000.0 / dt_raster

    sigma_samples = w / dt_raster # Sigma in samples of the raster

    # Apply Gaussian filter to each neuron's full-resolution binary spike train
    # Using convolve2d as per user request to avoid ndimage dependency issues
    k_size_gauss = int(7 * sigma_samples + 1)
    x_grid_gauss = jnp.linspace(-3.0 * sigma_samples, 3.0 * sigma_samples, k_size_gauss)
    kernel_1d_gauss = jnp.exp(-x_grid_gauss**2 / (2 * sigma_samples**2))
    kernel_1d_gauss = kernel_1d_gauss / jnp.sum(kernel_1d_gauss) # Normalize
    kernel_2d_gauss = kernel_1d_gauss[None, :]

    recorded_spiking_w = jax.scipy.signal.convolve2d(spike_image, kernel_2d_gauss, mode='same')

    # Compute the mean signal from recorded_spiking_w for spectrogram
    mean_signal_for_spectrogram = jnp.mean(recorded_spiking_w, axis=0)

    # Convert JAX numpy array to numpy array for scipy compatibility
    # And apply uniform filter (smoothing) as in original code for LFP proxy
    fs = fs_raster # Sampling frequency of the raster
    # smoothing_window_samples = int(1.0 / dt_global) # 1ms window for uniform filter
    mean_signal_np = np.asarray(mean_signal_for_spectrogram).astype(np.float64)
    # if smoothing_window_samples > 1:
    #     mean_signal_np = ndimage.uniform_filter1d(mean_signal_np, size=smoothing_window_samples)

    # Apply mirroring to the mean signal before spectrogram calculation
    nperseg = int(window_size * fs / 1000.0)
    if nperseg == 0: nperseg = 1
    half_window_samples = nperseg // 2

    extended_mean_signal_np = mean_signal_np
    if len(mean_signal_np) > half_window_samples:
        mirrored_prefix = mean_signal_np[1 : half_window_samples + 1][::-1]
        extended_mean_signal_np = np.concatenate((mirrored_prefix, extended_mean_signal_np))
    if len(mean_signal_np) > half_window_samples:
        mirrored_suffix = mean_signal_np[-(half_window_samples + 1) : -1][::-1]
        extended_mean_signal_np = np.concatenate((extended_mean_signal_np, mirrored_suffix))

    # Spectrogram parameters and computation
    noverlap = int(nperseg * overlap)
    if noverlap >= nperseg: noverlap = nperseg - 1
    if noverlap < 0: noverlap = 0

    frequencies, times, Sxx = signal.spectrogram(extended_mean_signal_np, fs=fs,
                                                 window='hann', nperseg=nperseg,
                                                 noverlap=noverlap)

    # Adjust the times for plotting to align with the original time_axis
    time_shift_ms = half_window_samples * dt_raster
    adjusted_times = times * 1000 - time_shift_ms

    freq_indices = (frequencies >= f_min) & (frequencies <= f_max)
    frequencies_filtered = frequencies[freq_indices]
    Sxx_filtered = (Sxx[freq_indices, :])

    # Apply aperiodic correction
    if aperiodic_correct != 0.0 and frequencies_filtered.size > 0:
        # Add a small epsilon to avoid division by zero for frequency=0
        Sxx_filtered = Sxx_filtered / (frequencies_filtered[:, jnp.newaxis]**aperiodic_correct + 1e-9)

    # Apply baseline relative normalization if requested
    if baseline_relative is not None and frequencies_filtered.size > 0:
        t_start_ms, t_end_ms = baseline_relative
        # Convert ms to time indices for adjusted_times
        t_start_idx = jnp.argmin(jnp.abs(adjusted_times - t_start_ms))
        t_end_idx = jnp.argmin(jnp.abs(adjusted_times - t_end_ms))

        # Ensure indices are valid and in order
        if t_start_idx > t_end_idx:
            t_start_idx, t_end_idx = t_end_idx, t_start_idx
        # Ensure the slice is not empty
        if t_start_idx == t_end_idx and t_end_idx < Sxx_filtered.shape[1] -1:
             t_end_idx +=1 # Extend slice to at least 2 points
        elif t_start_idx == t_end_idx and t_end_idx > 0:
             t_start_idx -=1 # Extend slice to at least 2 points

        baseline_power_per_freq = jnp.mean(Sxx_filtered[:, t_start_idx:t_end_idx + 1], axis=1, keepdims=True)
        # Avoid division by zero
        Sxx_filtered = Sxx_filtered / (baseline_power_per_freq + 1e-9)

    if frequencies_filtered.size == 0:
        print("Warning: No frequencies found within the specified f_min and f_max range for spectrogram in combined plot.")
    else:

        n_shape_sxx = int(Sxx_filtered.shape[1]*0.8)
        min_val = jnp.min(Sxx_filtered[:n_shape_sxx])
        max_val = jnp.max(Sxx_filtered[:n_shape_sxx])
        Sxx_scaled = (Sxx_filtered - min_val) / (max_val - min_val) if (max_val - min_val) != 0 else jnp.zeros_like(Sxx_filtered)
        Sxx_scaled = jnp.sqrt(Sxx_scaled)

        plt.imshow(Sxx_scaled, aspect='auto', cmap='jet', origin='lower',
                   extent=[adjusted_times[0], adjusted_times[-1], frequencies_filtered[0], frequencies_filtered[-1]])
        plt.colorbar(label='Normalized Power (0-1)')
        plt.title(f'Mean Spectrogram of Individual Neuronal Potentials (Convolved Spikes) {title_suffix}')
        plt.xlabel('Time (ms)')
        plt.ylabel('Frequency (Hz)')
        plt.ylim([frequencies_filtered[0], frequencies_filtered[-1]])
        plt.xlim([time_axis[0], time_axis[-1]])
        plt.axvline(x=500, color='red', linestyle='--', linewidth=1.5)
        red_line_freqs = [3.0, 8.0, 12.0, 20.0, 30.0, 70.0]
        for freq_line in red_line_freqs:
            if frequencies_filtered[0] <= freq_line <= frequencies_filtered[-1]:
                plt.axhline(y=freq_line, color='red', linestyle='--', linewidth=1.0)

    # Subplot 3: Average PSD Interval 1
    plt.subplot(4, 1, 3)
    if frequencies_filtered.size > 0:
        t_mask1 = (adjusted_times >= interval1[0]) & (adjusted_times <= interval1[1])
        if np.any(t_mask1):
            avg_psd1 = jnp.mean(Sxx_filtered[:, t_mask1], axis=1)
            min_psd1 = jnp.min(avg_psd1)
            max_psd1 = jnp.max(avg_psd1)
            avg_psd1_scaled = (avg_psd1 - min_psd1) / (max_psd1 - min_psd1) if (max_psd1 - min_psd1) != 0 else jnp.zeros_like(avg_psd1)

            plt.plot(frequencies_filtered, avg_psd1_scaled, color='blue')
            plt.title(f'Average PSD ({interval1[0]}-{interval1[1]} ms) {title_suffix}')
            plt.ylabel('Norm Power')
            plt.xlim([frequencies_filtered[0], frequencies_filtered[-1]])
            plt.ylim([0, 1.1])
            for freq_line in red_line_freqs:
                if frequencies_filtered[0] <= freq_line <= frequencies_filtered[-1]:
                    plt.axvline(x=freq_line, color='red', linestyle='--', linewidth=1.0)

    # Subplot 4: Average PSD Interval 2
    plt.subplot(4, 1, 4)
    if frequencies_filtered.size > 0:
        t_mask2 = (adjusted_times >= interval2[0]) & (adjusted_times <= interval2[1])
        if np.any(t_mask2):
            avg_psd2 = jnp.mean(Sxx_filtered[:, t_mask2], axis=1)
            min_psd2 = jnp.min(avg_psd2)
            max_psd2 = jnp.max(avg_psd2)
            avg_psd2_scaled = (avg_psd2 - min_psd2) / (max_psd2 - min_psd2) if (max_psd2 - min_psd2) != 0 else jnp.zeros_like(avg_psd2)

            plt.plot(frequencies_filtered, avg_psd2_scaled, color='blue')
            plt.title(f'Average PSD ({interval2[0]}-{interval2[1]} ms) {title_suffix}')
            plt.xlabel('Frequency (Hz)')
            plt.ylabel('Norm Power')
            plt.xlim([frequencies_filtered[0], frequencies_filtered[-1]])
            plt.ylim([0, 1.1])
            for freq_line in red_line_freqs:
                if frequencies_filtered[0] <= freq_line <= frequencies_filtered[-1]:
                    plt.axvline(x=freq_line, color='red', linestyle='--', linewidth=1.0)

    plt.tight_layout()

    # No separate `subplots_adjust` needed for `right` anymore if using `add_axes` for cbar, just for overall spacing.
    plt.subplots_adjust(right=0.9)

    if save:
        plt.savefig(savename, format='svg')

    plt.show()


### Network

#### EI-net

In [ ]:
def net_eig(num_e: int, num_ig: int, num_il: int):
    """
    Constructs a JAXley neural network with specified numbers of excitatory and inhibitory neurons.

    Args:
        num_e (int): Number of Excitatory neurons.
        num_ig (int): Number of Global inhibitory interneurons (SST-like).
        num_il (int): Number of Local inhibitory interneurons (PV-like).

    Returns:
        jx.Network: The constructed JAXley network object.
    """
    comp = jx.Compartment()
    branch = jx.Branch(comp, ncomp=2) # Changed to ncomp=2 for distinct pre/post branches

    e_cells = [jx.Cell(branch, parents=[-1, 0]) for _ in range(num_e)]
    i_cells_g = [jx.Cell(branch, parents=[-1, 0]) for _ in range(num_ig)]
    i_cells_l = [jx.Cell(branch, parents=[-1, 0]) for _ in range(num_il)]

    for cell in e_cells + i_cells_g + i_cells_l:
        r_d_tau_diff = np.random.uniform(10.0, 50.0)
        r_amp_diff = np.clip(np.random.uniform(-0.1, 0.1), 0.0, 0.05)
        cell.insert(HH())
        cell.insert(Inoise(initial_amp_noise=r_amp_diff, initial_tau=r_d_tau_diff, initial_mean=0.0))
        cell.radius = 1.0  # uniform constant geometry
        cell.length = 1.0

    net = jx.Network(e_cells + i_cells_g + i_cells_l)
    net.cell(np.arange(0, num_e)).add_to_group("E")
    net.cell(np.arange(num_e,num_e+num_ig)).add_to_group("Ing")
    net.cell(np.arange(num_e+num_ig,num_e+num_ig+num_il)).add_to_group("Inl")

    # 3. Connectivity (All-to-All between populations)
    # E -> all (AMPA)
    pre = net.cell([ik for ik in range(num_e)]).branch(0).loc(0.0) # From branch 0, loc 0.0
    post = net.cell([ik for ik in range(num_e+num_ig+num_il)]).branch(1).loc(1.0) # To branch 1, loc 1.0
    fully_connect(pre, post, GradedAMPA(tauD_AMPA=2.0))

    # Ig -> all (GABAa)
    pre = net.cell([ik for ik in range(num_e, num_e+num_ig)]).branch(0).loc(0.0)
    post = net.cell([ik for ik in range(num_e+num_ig+num_il)]).branch(1).loc(1.0)
    fully_connect(pre, post, GradedGABAa(tauD_GABAa=5.0))

    # Il -> subset of all (GABAa)
    pre_il = net.cell([ik for ik in range(num_e+num_ig, num_e+num_ig+num_il)]).branch(0).loc(0.0)
    num_posts_to_select = int((num_e+num_ig+num_il)*0.1)
    posts_pool_indices = jnp.arange(0, num_e + num_ig + num_il)
    key_conn = jax.random.PRNGKey(1)
    selected_post_indices = np.array(jax.random.choice(key_conn, posts_pool_indices, shape=(num_posts_to_select,), replace=False))
    post_il = net.cell(selected_post_indices).branch(1).loc(1.0) # Connect to branch 1
    fully_connect(pre_il, post_il, GradedGABAa(tauD_GABAa=2.0))

    # net.compute_xyz()
    # net.arrange_in_layers(layers=[num_e + num_ig + num_il], within_layer_offset=100.0, between_layer_offset=100.0)
    # fig, ax = plt.subplots(1, 1, figsize=(6, 12))
    # _ = net.vis(ax=ax, detail="full")

    return net


### Parameter sweep

In [ ]:
import pickle
import jax.numpy as jnp
import numpy as np
import jax.random as jr


def sw_simulate(net, this_params, single_input_amp):

    dt_global = 0.1
    t_max_vs = 1500.0
    t_onset_vs = 500.0
    t_dur_vs = 500.0

    levels = 2
    time_points_vs = t_max_vs / dt_global + 1
    checkpoints_vs = [int(np.ceil(time_points_vs**(1/levels))) for _ in range(levels)]

    ac_current = noise_current_ac(
        i_delay=t_onset_vs,
        i_dur=t_dur_vs,
        amp_n=0.0,
        amp_b=single_input_amp,
        delta_t=dt_global,
        t_max=t_max_vs,
        spect=jnp.array([120.0])
    )

    sim_current = ac_current

    net.delete_stimuli()
    net.delete_recordings()
    num_posts_to_select = int(Ne/2)
    posts_pool_indices = jnp.concat([jnp.arange(0, Ne)])
    key_conn = jax.random.PRNGKey(5678)
    selected_post_indices = np.array(jax.random.choice(key_conn, posts_pool_indices, shape=(num_posts_to_select,), replace=False))

    data_stimuli_single_run = net.cell(selected_post_indices).branch(1).loc(0.0).data_stimulate(sim_current) # Changed loc(1.0) to loc(0.0)
    net.cell([i for i in range(Ne+Nig+Nil)]).branch(0).loc(0.0).record()

    print(f"Running simulation with input amplitude: {single_input_amp:.2f} nA...")
    recorded_voltages_pre_training = jx.integrate(net, params=this_params, data_stimuli=data_stimuli_single_run, checkpoint_lengths=checkpoints_vs)
    print(f"Simulation completed. Recorded voltages shape: {recorded_voltages_pre_training.shape}")

    return recorded_voltages_pre_training


def save_logs(voltages, indices, losses, path):
    with open(path, 'wb') as f:
        pickle.dump({'voltages': voltages, 'indices': indices, 'losses': losses}, f)
    print(f"Logs saved to {path}")

def load_logs(path):
    with open(path, 'rb') as f:
        data = pickle.load(f)
    print(f"Logs loaded from {path}")
    return data['voltages'], data['indices'], data['losses']


def get_edge_indices(net, source_cells, target_cells):

    if 'pre_cell_index' in net.edges.columns and 'post_cell_index' in net.edges.columns:
        pre_cell_ids_for_edges = net.edges['pre_cell_index'].values
        post_cell_ids_for_edges = net.edges['post_cell_index'].values
    else:
        # Fallback: manually map node indices to cell indices
        cell_id_map = net.nodes['local_cell_index'].values
        pre_node_indices = net.edges['pre_index'].values
        post_node_indices = net.edges['post_index'].values

        pre_cell_ids_for_edges = np.full(len(pre_node_indices), -1)
        post_cell_ids_for_edges = np.full(len(post_node_indices), -1)
        pre_valid_indices = pre_node_indices < len(cell_id_map)
        post_valid_indices = post_node_indices < len(cell_id_map)

        pre_cell_ids_for_edges[pre_valid_indices] = cell_id_map[pre_node_indices[pre_valid_indices]]
        post_cell_ids_for_edges[post_valid_indices] = cell_id_map[post_node_indices[post_valid_indices]]

    # Ensure source_cells and target_cells are flat arrays for np.isin
    edge_indices_pre = np.where(np.isin(pre_cell_ids_for_edges, np.array(source_cells).flatten()))[0]
    edge_indices_post = np.where(np.isin(post_cell_ids_for_edges, np.array(target_cells).flatten()))[0]

    edge_indices = np.intersect1d(edge_indices_pre, edge_indices_post)

    return edge_indices


Ne = 18
Nig = 4
Nil = 3

net = net_eig(Ne, Nig, Nil)
net.GradedAMPA.edge("all").make_trainable("gAMPA")
net.GradedGABAa.edge("all").make_trainable("gGABAa")

cell_group_e = np.arange(Ne)
cell_group_i = np.arange(Ne, Ne+Nig+Nil)

In [ ]:
logs_save_path = os.path.join(save_dir, "simulation_logs_e2_i2.pkl")

import jax.random as jr
key = jr.PRNGKey(0)

log_voltages = []
log_voltages_ix = []
log_loss = []
step_d = 0.1

for ik in range(100):
    print(ik)
    for jk in range(100):
        lb_temp1 = ik*step_d
        ub_temp1 = ik*step_d + 1.0
        lb_temp2 = jk*step_d
        ub_temp2 = jk*step_d + 1.0

        this_params = net.get_parameters()

        # Use JAX random and JAX numpy for parameter generation and clipping
        key, subkey1, subkey2 = jr.split(key, 3)
        random_ampa_values = jr.uniform(subkey1, shape=this_params[0]['gAMPA'].shape, minval=-1.0, maxval=ub_temp1)
        this_params[0]['gAMPA'] = jnp.clip(random_ampa_values, lb_temp1, ub_temp1)

        random_gaba_values = jr.uniform(subkey2, shape=this_params[1]['gGABAa'].shape, minval=-1.0, maxval=ub_temp2)
        this_params[1]['gGABAa'] = jnp.clip(random_gaba_values, lb_temp2, ub_temp2)

        recorded_voltages_pre_training = sw_simulate(net, this_params, inputs[1])

        log_voltages.append(recorded_voltages_pre_training)
        log_voltages_ix.append([ik, jk])
        log_loss.append(loss(this_params, inputs[1:2], labels[1:2]))
        print(f"Loss: {log_loss[-1]}")


save_logs(log_voltages, log_voltages_ix, log_loss, logs_save_path)
# loaded_log_voltages, loaded_log_voltages_ix, loaded_log_loss = load_logs(logs_save_path)
# print(f"Loaded log_voltages shape: {loaded_log_voltages[0].shape}")

In [ ]:
logs_save_path = os.path.join(save_dir, "simulation_logs_ee_ii.pkl")

log_voltages = []
log_voltages_ix = []
log_loss = []

ampa_ee = get_edge_indices(net, cell_group_e, cell_group_e)
gaba_ii = get_edge_indices(net, cell_group_i, cell_group_i)
# ampa_ei = get_edge_indices(net, cell_group_e, cell_group_i)
# gaba_ie = get_edge_indices(net, cell_group_i, cell_group_e)

key = jr.PRNGKey(0)

step_d = 0.1

for ik in range(100):
    print(ik)
    for jk in range(100):
        lb_temp1 = ik*step_d
        ub_temp1 = ik*step_d + 1.0
        lb_temp2 = jk*step_d
        ub_temp2 = jk*step_d + 1.0

        this_params = net.get_parameters()
        ampa_ee_jnp = jnp.asarray(ampa_ee)
        gaba_ii_jnp = jnp.asarray(gaba_ii)

        if ampa_ee_jnp.size > 0:
            key, subkey_ampa = jr.split(key)
            random_ampa_values = jr.uniform(subkey_ampa, shape=ampa_ee_jnp.shape, minval=-1.0, maxval=ub_temp1)
            clipped_ampa_values = jnp.clip(random_ampa_values, lb_temp1, ub_temp1)
            this_params[0]['gAMPA'] = this_params[0]['gAMPA'].at[ampa_ee_jnp].set(clipped_ampa_values)

        if gaba_ii_jnp.size > 0:
            key, subkey_gaba = jr.split(key)
            random_gaba_values = jr.uniform(subkey_gaba, shape=gaba_ii_jnp.shape, minval=-1.0, maxval=ub_temp2)
            clipped_gaba_values = jnp.clip(random_gaba_values, lb_temp2, ub_temp2)
            this_params[1]['gGABAa'] = this_params[1]['gGABAa'].at[gaba_ii_jnp].set(clipped_gaba_values)

        recorded_voltages_pre_training = sw_simulate(net, this_params, inputs[1])

        log_voltages.append(recorded_voltages_pre_training)
        log_voltages_ix.append([ik, jk])
        log_loss.append(loss(this_params, inputs[1:2], labels[1:2]))
        print(f"Loss: {log_loss[-1]}")


save_logs(log_voltages, log_voltages_ix, log_loss, logs_save_path)
# loaded_log_voltages, loaded_log_voltages_ix, loaded_log_loss = load_logs(logs_save_path)
# print(f"Loaded log_voltages shape: {loaded_log_voltages[0].shape}")

In [ ]:
logs_save_path = os.path.join(save_dir, "simulation_logs_ei_ie.pkl")


log_voltages = []
log_voltages_ix = []
log_loss = []

# ampa_ee = get_edge_indices(net, cell_group_e, cell_group_e)
# gaba_ii = get_edge_indices(net, cell_group_i, cell_group_i)
ampa_ei = get_edge_indices(net, cell_group_e, cell_group_i)
gaba_ie = get_edge_indices(net, cell_group_i, cell_group_e)

key = jr.PRNGKey(0)

ampa_ei_jnp = jnp.asarray(ampa_ei)
gaba_ie_jnp = jnp.asarray(gaba_ie)

step_d = 0.1

for ik in range(100):
    print(ik)
    for jk in range(100):
        lb_temp1 = ik*step_d
        ub_temp1 = ik*step_d + 1.0
        lb_temp2 = jk*step_d
        ub_temp2 = jk*step_d + 1.0

        this_params = net.get_parameters()

        if ampa_ei_jnp.size > 0:
            key, subkey_ampa = jr.split(key)
            random_ampa_values = jr.uniform(subkey_ampa, shape=ampa_ei_jnp.shape, minval=-1.0, maxval=ub_temp1)
            clipped_ampa_values = jnp.clip(random_ampa_values, lb_temp1, ub_temp1)
            this_params[0]['gAMPA'] = this_params[0]['gAMPA'].at[ampa_ei_jnp].set(clipped_ampa_values)

        if gaba_ie_jnp.size > 0:
            key, subkey_gaba = jr.split(key)
            random_gaba_values = jr.uniform(subkey_gaba, shape=gaba_ie_jnp.shape, minval=-1.0, maxval=ub_temp2)
            clipped_gaba_values = jnp.clip(random_gaba_values, lb_temp2, ub_temp2)
            this_params[1]['gGABAa'] = this_params[1]['gGABAa'].at[gaba_ie_jnp].set(clipped_gaba_values)

        recorded_voltages_pre_training = sw_simulate(net, this_params, inputs[1])

        log_voltages.append(recorded_voltages_pre_training)
        log_voltages_ix.append([ik, jk])
        log_loss.append(loss(this_params, inputs[1:2], labels[1:2]))
        print(f"Loss: {log_loss[-1][0]}")


save_logs(log_voltages, log_voltages_ix, log_loss, logs_save_path)

In [ ]:
logs_save_path = os.path.join(save_dir, "simulation_logs_2e_2i.pkl")


log_voltages = []
log_voltages_ix = []
log_loss = []

ampa_ee = get_edge_indices(net, cell_group_e, cell_group_e)
gaba_ii = get_edge_indices(net, cell_group_i, cell_group_i)
ampa_ei = get_edge_indices(net, cell_group_e, cell_group_i)
gaba_ie = get_edge_indices(net, cell_group_i, cell_group_e)

key = jr.PRNGKey(0)

step_d = 0.1

for ik in range(100):
    print(ik)
    for jk in range(100):
        lb_temp1 = ik*step_d
        ub_temp1 = ik*step_d + 1.0
        lb_temp2 = jk*step_d
        ub_temp2 = jk*step_d + 1.0

        this_params = net.get_parameters()
        ampa_ei_jnp = jnp.asarray(ampa_ei)
        gaba_ie_jnp = jnp.asarray(gaba_ie)
        ampa_ee_jnp = jnp.asarray(ampa_ee)
        gaba_ii_jnp = jnp.asarray(gaba_ii)

        if ampa_ei_jnp.size > 0:
            key, subkey_ampa = jr.split(key)
            random_ampa_values = jr.uniform(subkey_ampa, shape=ampa_ei_jnp.shape, minval=-1.0, maxval=ub_temp1)
            clipped_ampa_values = jnp.clip(random_ampa_values, lb_temp1, ub_temp1)
            this_params[0]['gAMPA'] = this_params[0]['gAMPA'].at[ampa_ei_jnp].set(clipped_ampa_values)
        if gaba_ii_jnp.size > 0:
            key, subkey_gaba = jr.split(key)
            random_gaba_values = jr.uniform(subkey_gaba, shape=gaba_ii_jnp.shape, minval=-1.0, maxval=ub_temp1)
            clipped_gaba_values = jnp.clip(random_gaba_values, lb_temp1, ub_temp1)
            this_params[1]['gGABAa'] = this_params[1]['gGABAa'].at[gaba_ii_jnp].set(clipped_gaba_values)
        if gaba_ie_jnp.size > 0:
            key, subkey_gaba = jr.split(key)
            random_gaba_values = jr.uniform(subkey_gaba, shape=gaba_ie_jnp.shape, minval=-1.0, maxval=ub_temp2)
            clipped_gaba_values = jnp.clip(random_gaba_values, lb_temp2, ub_temp2)
            this_params[1]['gGABAa'] = this_params[1]['gGABAa'].at[gaba_ie_jnp].set(clipped_gaba_values)
        if ampa_ee_jnp.size > 0:
            key, subkey_ampa = jr.split(key)
            random_ampa_values = jr.uniform(subkey_ampa, shape=ampa_ee_jnp.shape, minval=-1.0, maxval=ub_temp2)
            clipped_ampa_values = jnp.clip(random_ampa_values, lb_temp2, ub_temp2)
            this_params[0]['gAMPA'] = this_params[0]['gAMPA'].at[ampa_ee_jnp].set(clipped_ampa_values)

        recorded_voltages_pre_training = sw_simulate(net, this_params, inputs[1])

        log_voltages.append(recorded_voltages_pre_training)
        log_voltages_ix.append([ik, jk])
        log_loss.append(loss(this_params, inputs[1:2], labels[1:2]))
        print(f"Loss: {log_loss[-1]}")


save_logs(log_voltages, log_voltages_ix, log_loss, logs_save_path)

In [ ]:
all_loaded_data = []
save_dir_sweep = "/content/data/Data"

for file_name in os.listdir(save_dir_sweep):
    if file_name.startswith('simulation_logs_') and file_name.endswith('.pkl'):
        full_file_path = os.path.join(save_dir_sweep, file_name)
        try:
            # load_logs returns voltages, indices, losses
            voltages, indices, losses = load_logs(full_file_path)
            all_loaded_data.append({
                'file_name': file_name,
                'log_loss': losses,
                'log_voltages_ix': indices,
                'log_voltages': voltages # Store voltages if needed for future subtasks
            })
        except Exception as e:
            print(f"Error loading {file_name}: {e}")

print(f"Loaded {len(all_loaded_data)} simulation log files.")

In [ ]:
zoom_factor = 100 # Upsample to 1000x1000 (100 * 10) for smoother visualization
sigma = 5.0       # Gaussian smoothing factor

for data_entry in all_loaded_data:
    file_name = data_entry['file_name']
    log_loss = data_entry['log_loss']
    log_voltages_ix = data_entry['log_voltages_ix']

    # Initialize a 10x10 NumPy array to store the loss values
    loss_grid = np.zeros((10, 10))

    for index in range(len(log_loss)):
        # Ensure loss_value is extracted correctly (first element of tuple if it's a tuple)
        loss_value = log_loss[index]
        if isinstance(loss_value, tuple):
            loss_value = loss_value[0] # Take the first element if it's a tuple

        # Extract ik and jk indices
        ik, jk = log_voltages_ix[index]

        # Assign loss value to the correct position in loss_grid
        loss_grid[ik, jk] = loss_value

    # Upsample the loss grid
    upsampled_loss_grid = ndimage.zoom(loss_grid, zoom_factor, order=3)

    # Apply Gaussian smoothing
    smoothed_loss_grid = ndimage.gaussian_filter(upsampled_loss_grid, sigma=sigma)

    # Plot the smoothed data as a heatmap
    plt.figure(figsize=(10, 8))
    plt.imshow(smoothed_loss_grid, cmap='turbo', origin='lower', extent=[0, 10, 0, 10])
    plt.colorbar(label='Loss Value')
    plt.title(f'Smoothed Loss Grid Heatmap for: {file_name}')
    plt.xlabel('ik (x-axis)')
    plt.ylabel('jk (y-axis)')
    plt.xlim(0, 10)
    plt.ylim(0, 10)
    plt.grid(False)

    # Find the minimum value and its coordinates in the smoothed grid
    min_value = np.min(smoothed_loss_grid)
    min_indices = np.unravel_index(np.argmin(smoothed_loss_grid, axis=None), smoothed_loss_grid.shape)

    # Convert min_indices from upsampled grid coordinates to original 0-10 scale
    min_x = min_indices[1] / zoom_factor + (0.5 / zoom_factor) # Add half a bin to center
    min_y = min_indices[0] / zoom_factor + (0.5 / zoom_factor)

    # Plot a star at the minimum location
    plt.plot(min_x, min_y + 0.2, 'w*', markersize=15, markeredgecolor='black', label='Minimum Loss')

    # Add text annotation for the minimum value
    plt.text(min_x - 0.2, min_y - 0.2, f'{min_value:.2f}', color='white', fontsize=12, fontweight='bold', ha='left', va='center')

    # Construct filename and save the figure
    base_file_name = os.path.splitext(file_name)[0]
    save_file_path = os.path.join(figure_save_path, f'{base_file_name}_loss_heatmap.svg')
    plt.savefig(save_file_path)

    plt.show()

In [ ]:
# Ne = 36
# Nig = 7
# Nil = 7

# net = net_eig(Ne, Nig, Nil)

# net.edges.gAMPA = np.clip(np.random.uniform(-5.0, 10.0, net.edges.gAMPA.shape), 2.0, 5.0)
# net.edges.gGABAa = np.clip(np.random.uniform(-5.0, 10.0, net.edges.gGABAa.shape), 4.0, 10.0)

# net.set("gAMPA", 2.5)
# net.set("gGABAa", 5.0)
# net.cell([i for i in range(Ne, Ne + Nig + Nil)]).set("amp_noise", 0.02)
# net.cell([i for i in range(Ne, Ne + Nig + Nil)]).set("mean", 0.0)
# net.cell([i for i in range(Ne)]).set("amp_noise", 0.02)
# net.cell([i for i in range(Ne)]).set("mean", 0.0)

dt_global = 0.1
t_max_vs = 1500.0
t_onset_vs = 500.0
t_dur_vs = 500.0
single_input_amp = inputs[1]*6/6

levels = 2
time_points_vs = t_max_vs / dt_global + 1
checkpoints_vs = [int(np.ceil(time_points_vs**(1/levels))) for _ in range(levels)]

ac_current = noise_current_ac(
    i_delay=t_onset_vs,
    i_dur=t_dur_vs,
    amp_n=0.0,
    amp_b=single_input_amp,
    delta_t=dt_global,
    t_max=t_max_vs,
    spect=jnp.array([120.0])
)

sim_current = ac_current

this_params = net.get_parameters()

net.delete_stimuli()
net.delete_recordings()
num_posts_to_select = int(Ne/2)
posts_pool_indices = jnp.concat([jnp.arange(0, Ne)])
key_conn = jax.random.PRNGKey(5678)
selected_post_indices = np.array(jax.random.choice(key_conn, posts_pool_indices, shape=(num_posts_to_select,), replace=False))

data_stimuli_single_run = net.cell(selected_post_indices).branch(1).loc(0.0).data_stimulate(sim_current) # Changed loc(1.0) to loc(0.0)
net.cell([i for i in range(Ne+Nig+Nil)]).branch(0).loc(0.0).record()

print(f"Running simulation with input amplitude: {single_input_amp:.2f} nA...")
recorded_voltages_pre_training = jx.integrate(net, params=this_params, data_stimuli=data_stimuli_single_run, checkpoint_lengths=checkpoints_vs)
print(f"Simulation completed. Recorded voltages shape: {recorded_voltages_pre_training.shape}")
time_axis = jnp.arange(20, t_max_vs, dt_global)

trace_vis_tfr(recorded_voltages_pre_training)

In [ ]:
trace_vis_tfr(recorded_voltages_pre_training)

# Spectral tuning

### Data

In [ ]:
def get_signal_psd(traces, fs=1000):
  signal = jnp.nanmean(jnp.squeeze(traces), axis=1) # Average voltage across all recorded neurons over time
  # signal = ndimage.uniform_filter1d(signal, size=10)
  N = signal.shape[-1]

  # Compute one-sided FFT and corresponding frequencies for real signals
  signal_fft = jnp.fft.rfft(signal)
  freqs = jnp.fft.rfftfreq(N, 1/fs)
  psd_raw = jnp.abs(signal_fft)**2 / (N * fs)

  target_freqs = global_psd_interval
  interpolated_psd = jnp.interp(target_freqs, freqs, psd_raw)
  interpolated_psd = ndimage.uniform_filter1d(interpolated_psd, size=jnp.ceil(5*jnp.sqrt(fs/1000)).astype(int))

  interpolated_psd = interpolated_psd - jnp.min(interpolated_psd)
  max_psd = jnp.max(interpolated_psd)
  scaled_psd = interpolated_psd / (max_psd + 1e-6)
  return scaled_psd


global_psd_interval = jnp.linspace(15.0, 60.0, 40)
nbin_freq = global_psd_interval.shape[0]
inputs = jnp.array([0.0, 1.0, 1.0])*0.6 # nA stim amp for the three groups
labels = jnp.zeros((inputs.shape[0], nbin_freq))

for ik in range(1):
  base_units_spectrum = get_signal_psd(jnp.nanmean(selected_unit_spiking[5:6, :200, :], 0))
  labels = labels.at[ik].set(base_units_spectrum)

for ik in range(1, 3):
  stim_units_spectrum = get_signal_psd(jnp.nanmean(selected_unit_spiking[5:6, 600:1200, :], 0))
  labels = labels.at[ik].set(stim_units_spectrum)

batch_size = 3
dataloader = Dataset(inputs, labels)
dataloader = dataloader.shuffle(seed=0).batch(batch_size)

# Visualization of the labels (PSDs)
plt.figure(figsize=(10, 6))
freq_axis = global_psd_interval

for i in range(0, labels.shape[0]):
    label_name = f"Baseline" if i < 1 else f"Stim on"
    plt.plot(freq_axis, labels[i], label=label_name)

plt.title("Target Power Spectral Densities (Labels)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Normalized Power")
plt.xlim([5, 60])
plt.ylim([0, 1])
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# temp_trace = np.squeeze(np.mean(selected_unit_spiking[5:6, 0:1500, :], 0)).T
# # temp_trace.shape
# trace_vis_tfr(temp_trace, dt_trace=1.0)

In [ ]:
# Update the global variable so get_signal_psd uses the correct resolution (80 bins)
global_psd_interval = jnp.linspace(15.0, 60.0, 80)
nbin_freq_g = global_psd_interval.shape[0]

inputs = jnp.array([0.0, 1.0, 1.0])*0.6 # nA stim amp for the three groups
labels_g = jnp.zeros((10, nbin_freq_g))

for ik in range(10):
  # get_signal_psd will now use the updated global_psd_interval
  stim_units_spectrum_g = get_signal_psd(jnp.nanmean(selected_unit_spiking[ik*4:ik*4+4, 600:1200, :], 0))
  labels_g = labels_g.at[ik].set(stim_units_spectrum_g)

plt.figure(figsize=(10, 6))

# Use imshow to display as a heatmap
# extent sets the axis values: [x_min, x_max, y_min, y_max]
plt.imshow(labels_g, aspect='auto', origin='lower',
           extent=[global_psd_interval[0], global_psd_interval[-1], 0, 10])

plt.title("Target Power Spectral Densities (Labels)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Trial Group Index")
plt.colorbar(label='Normalized Power')
plt.tight_layout()
plt.show()

In [ ]:
jnp.mean(selected_unit_spiking[5:10, 600:1100, :], 0).shape
selected_unit_spiking.shape

In [ ]:
current_amp_nA = 1.0
currents = noise_current(
    i_delay=t_on_global,
    i_dur=500.0,
    i_amp=current_amp_nA,
    delta_t=dt_global,
    t_max=t_max_global
)

step_currents = step_current(
    i_delay=t_on_global,
    i_dur=500.0,
    i_amp=current_amp_nA,
    delta_t=dt_global,
    t_max=t_max_global
)

sim_current = noise_current_ac(
    i_delay=t_onset_vs,
    i_dur=t_dur_vs,
    amp_n=0.0,
    amp_b=1.0,
    delta_t=dt_global,
    t_max=t_max_vs,
    spect=jnp.array([120.0])
)
# sim_current = step_currents + currents
plt.plot(sim_current)

## Model initilaizations

In [ ]:
band_definitions = {
    'gamma': (35.0, 60.0),
    'beta': (13.0, 30.0),
    'alpha': (8.0, 12.0),
    'theta': (4.0, 8.0)
}

lower_c = 5.0
upper_c = 50.0
firing_rate_weight = 9.0
psd_weight = 990.0

dt_global = 0.1
t_max_global = 1500
t_on_global = 500

Ne = 36 # Number of Excitatory neurons
Nig = 8 # Number of Global inhibitory interneurons (SST-like)
Nil = 6 # Number of Local inhibitory interneurons (PV-like))
net = net_eig(Ne, Nig, Nil)

net.edges.gAMPA = np.clip(np.random.uniform(-5.0, 10.0, net.edges.gAMPA.shape), 0, 10.0)
net.edges.gGABAa = np.clip(np.random.uniform(-1.0, 10.0, net.edges.gGABAa.shape), 0, 10.0)

gAMPA_lower_bounds = 0.00
gAMPA_upper_bounds = 10.00

gGABA_lower_bounds = 0.00
gGABA_upper_bounds = 10.00

# Correctly define transform as ParamTransform for the entire parameter tree
transform = jx.ParamTransform([
    {"gAMPA": ClampTransform(gAMPA_lower_bounds, gAMPA_upper_bounds)},
    {"gGABAa": ClampTransform(gGABA_lower_bounds, gGABA_upper_bounds)}
])

net.delete_recordings()
net.delete_stimuli()
net.delete_trainables()

net.GradedAMPA.edge("all").make_trainable("gAMPA")
net.GradedGABAa.edge("all").make_trainable("gGABAa")
view_ampa = net.GradedAMPA.edge("all")
view_gaba = net.GradedGABAa.edge("all")

# Helper to safely extract column
def get_edge_indices(view, col_candidates):
    for col in col_candidates:
        if col in view.edges.columns:
            return jnp.array(view.edges[col].values.astype(int))
    # If not found, print available columns for debugging and return empty array
    print(f"DEBUG: No candidate column found. Available columns: {list(view.edges.columns)}")
    return jnp.array([]) # Return an empty JAX array instead of raising an error

pre_candidates = ["pre_cell_idx", "pre_cell_index", "pre", "source", "pre_index"]
post_candidates = ["post_cell_idx", "post_cell_index", "post", "target", "post_index"]

ampa_pre_inds = get_edge_indices(view_ampa, pre_candidates)
ampa_post_inds = get_edge_indices(view_ampa, post_candidates)
gaba_pre_inds = get_edge_indices(view_gaba, pre_candidates)
gaba_post_inds = get_edge_indices(view_gaba, post_candidates)


@jit
def compute_correlations(traces, pre_inds, post_inds):
    """
    Computes Mutual Pearson correlation (est., time adjusted) between pre and post traces.
    traces: (Batch, Cells, Time)
    pre_inds, post_inds: (Num_Edges,)
    Returns: (Num_Edges,)
    """
    if pre_inds.size == 0 or post_inds.size == 0:
        return jnp.array([])

    # 1. Compute Raw Correlations
    pre_v = traces[:, pre_inds, :-10]
    post_v = traces[:, post_inds, 10:]

    pre_mean = jnp.mean(pre_v, axis=-1, keepdims=True)
    post_mean = jnp.mean(post_v, axis=-1, keepdims=True)
    pre_c = pre_v - pre_mean
    post_c = post_v - post_mean

    num = jnp.sum(pre_c * post_c, axis=-1)
    den = jnp.sqrt(jnp.sum(pre_c**2, axis=-1) * jnp.sum(post_c**2, axis=-1))
    corr = num**2 / (den + 1e-6) # (Batch, Num_Edges)
    num_total_neurons = traces.shape[1]

    k = int(np.sqrt(num_total_neurons))
    x_grid = jnp.linspace(-2.0, 2.0, k)
    window = jnp.exp(-x_grid**2 / 2.)
    kernel = jnp.outer(window, window)
    kernel = kernel / jnp.sum(kernel)  # Normalize

    def smooth_single_batch(c_flat):
        # Map sparse edges to dense matrix (N_cells, N_cells)
        mat = jnp.zeros((num_total_neurons, num_total_neurons))
        mat = mat.at[pre_inds, post_inds].set(c_flat)

        # Convolve with 2D kernel
        smoothed_mat = jax.scipy.signal.convolve2d(mat, kernel, mode='same')

        # Extract values back to sparse edge list
        return smoothed_mat[pre_inds, post_inds]

    corr_smoothed = jax.vmap(smooth_single_batch)(corr)

    return jnp.mean(corr_smoothed, axis=0)


def calculate_mcdp(traces):
    """
    Calculates normalized MCDP factors for all trainable parameters.
    """
    # Compute raw correlations
    r_ampa = compute_correlations(traces, ampa_pre_inds, ampa_post_inds)
    r_gaba = compute_correlations(traces, gaba_pre_inds, gaba_post_inds)

    # Normalize: (val - mean) / std
    # Note: Normalization is per parameter group
    # Handle cases where r_ampa or r_gaba might be empty
    r_ampa_norm = (r_ampa - jnp.mean(r_ampa)) / (jnp.std(r_ampa) + 1e-6) if r_ampa.size > 0 else jnp.array([])
    r_gaba_norm = (r_gaba - jnp.mean(r_gaba)) / (jnp.std(r_gaba) + 1e-6) if r_gaba.size > 0 else jnp.array([])

    # Return structure matching the params PyTree
    # [{gAMPA: ...}, {gGABAa: ...}]
    return [{'gAMPA': r_ampa_norm}, {'gGABAa': r_gaba_norm}] # , {'amp_noise': jnp.array([0])}


levels = 2
time_points = t_max_global / dt_global + 1
checkpoints = [int(np.ceil(time_points**(1/levels))) for _ in range(levels)]

net.delete_recordings()
net.cell([i for i in range(Ne+Nig+Nil)]).branch(0).loc(0.0).record()

def calculate_firing_rates(traces, spike_threshold=-20.0):
    """
    Calculates the average firing rate (spikes/second) for each neuron in the given voltage traces.

    Args:
        traces (jnp.ndarray): Array of voltage traces with shape (num_batches, num_neurons, num_timepoints).
                              The first column of each neuron's trace is assumed to be V_init and skipped.
        spike_threshold (float): The voltage threshold for detecting a spike (in mV).

    Returns:
        jnp.ndarray: Array of average firing rates (in Hz) for each neuron, with shape (num_batches, num_neurons).
    """

    # Determine the actual simulation duration excluding the initial V_init point
    num_timepoints_actual = traces.shape[-1] - 1
    # Total simulation time in milliseconds (since t_max_global is in ms)
    sim_duration_ms = t_max_global
    # Total simulation time in seconds
    sim_duration_s = sim_duration_ms / 1000.0

    def detect_spikes_for_neuron(neuron_trace):
        # Exclude the initial V_init point from the trace
        neuron_trace_data = neuron_trace[1:]
        # Detect upward crossings of the spike_threshold
        spikes = (neuron_trace_data[:-1] < spike_threshold) & (neuron_trace_data[1:] >= spike_threshold)
        return jnp.sum(spikes)

    # Vmap over neurons to get spike counts for all neurons in a single batch
    # input_axes=(0, None) means the first axis of neuron_traces is mapped, second (spike_threshold) is broadcast
    batched_detect_spikes_for_neurons = vmap(detect_spikes_for_neuron, in_axes=(0))

    # Vmap over batches to get spike counts for all batches
    # input_axes=(0) means the first axis of traces (batches) is mapped
    total_spike_counts_per_neuron_per_batch = vmap(batched_detect_spikes_for_neurons, in_axes=(0))(traces)

    # Calculate firing rate (spikes/second)
    firing_rates_hz = total_spike_counts_per_neuron_per_batch / sim_duration_s

    return firing_rates_hz


def simulate(params, input_amp):
    """
    Simulates the network for a given scalar input amplitude.
    """
    current_amp_nA = input_amp
    ac_currents = noise_current_ac(
        i_delay=t_on_global,
        i_dur=500.0,
        amp_n=0.0,
        amp_b=current_amp_nA,
        delta_t=dt_global,
        t_max=t_max_global,
        spect=jnp.array([120.0])
    )

    net.delete_stimuli()
    data_stimuli = net.cell([ik for ik in range(0, Ne, 2)]).branch(1).loc(0.0).data_stimulate(ac_currents)

    # Integrate the network dynamics.
    traces = jx.integrate(net, params=params, data_stimuli=data_stimuli, checkpoint_lengths=checkpoints)
    return traces


batched_simulate = vmap(simulate, in_axes=(None, 0))


def predict(params, input_amp):
    """
    Predicts a scaled PSD array.
    """
    traces = simulate(params, input_amp)
    signal = jnp.mean(traces[:, 1:], axis=0)

    N = signal.shape[-1]
    fs = 1000.0 / dt_global

    signal_fft = jnp.fft.rfft(signal)
    freqs = jnp.fft.rfftfreq(N, d=dt_global/1000)
    psd_raw = jnp.abs(signal_fft)**2 / (N * fs)

    target_freqs = global_psd_interval
    interpolated_psd = jnp.interp(target_freqs, freqs, psd_raw)

    max_psd = jnp.max(interpolated_psd)
    scaled_psd = interpolated_psd / (max_psd + 1e-6)

    return scaled_psd


def loss(opt_params, inputs, labels):
    """
    Loss function for PSD fitting with firing rate penalty. Returns total loss and traces (auxiliary).
    """
    params = transform.forward(opt_params)

    # Get traces directly via batched_simulate to use for both loss and MCDP
    traces = batched_simulate(params, inputs) # Shape (Batch, Cells, Time)

    # Calculate PSDs from traces for loss
    # Helper to calculate PSD for a single trace (mean over cells)
    def compute_psd_from_trace(trace):
        # trace: (Cells, Time)
        temp_l = int(100/dt_global)
        temp_r = int(1400/dt_global)
        signal = jnp.mean(trace[:, temp_l:temp_r], axis=0)
        N = signal.shape[-1]
        fs = 1000.0 / dt_global
        signal_fft = jnp.fft.rfft(signal)
        freqs = jnp.fft.rfftfreq(N, d=dt_global/1000)
        psd_raw = jnp.abs(signal_fft)**2 / (N * fs)

        target_freqs = global_psd_interval
        interpolated_psd = jnp.interp(target_freqs, freqs, psd_raw)
        max_psd = jnp.max(interpolated_psd)
        return interpolated_psd / (max_psd + 1e-6)

    predictions = vmap(compute_psd_from_trace)(traces)

    epsilon = 1e-6
    # psd_loss_per_sample = jnp.sum(jnp.square(labels * jnp.log((predictions + epsilon) / (labels + epsilon))), axis=1)
    psd_loss_per_sample = jnp.sum(jnp.square(labels * jnp.square((predictions + epsilon) - (labels + epsilon))), axis=1)

    # Calculate firing rates
    firing_rates = calculate_firing_rates(traces)

    # Compute firing rate penalty
    # The penalty is mean(exp(lower_c - F) + exp(F - upper_c)) over all neurons and batches
    penalty_lower = jnp.exp(lower_c - firing_rates)
    penalty_upper = jnp.exp(firing_rates - upper_c)
    firing_rate_penalty = jnp.mean(penalty_lower + penalty_upper) # Mean over all neurons and batches
    # jax.debug.print("Firing Rate Penalty: {}", firing_rate_penalty)

    # Combine PSD loss and firing rate penalty
    total_loss = jnp.mean(psd_loss_per_sample) * psd_weight + firing_rate_weight * firing_rate_penalty

    return total_loss, traces


# JIT the loss function and its gradient for efficiency
# has_aux=True because loss now returns (value, auxiliary_data)
jitted_grad = jit(value_and_grad(loss, argnums=0, has_aux=True))


def compute_unscaled_psd_from_trace(trace, dt_global):
    # Assumes trace is 1D (mean signal over neurons) and already smoothed
    N = trace.shape[-1]
    fs = 1000.0 / dt_global  # dt_global is in ms, fs in Hz

    # trace is already the mean signal over neurons, 1D array of shape (Time,)
    signal = trace

    signal_fft = jnp.fft.rfft(signal)
    freqs = jnp.fft.rfftfreq(N, d=dt_global/1000)
    psd_raw = jnp.abs(signal_fft)**2 / (N * fs)

    target_freqs = jnp.linspace(1.0, 100.0, 100)
    interpolated_psd = jnp.interp(target_freqs, freqs, psd_raw)

    return target_freqs, interpolated_psd


def calculate_psd_bands(psd, freqs, band_defs):
    """
    Calculates the average absolute power within specified frequency bands from PSD data.
    Handles multi-dimensional PSD arrays (e.g. Batch x Cells x Freqs) by averaging.
    Returns a list of powers corresponding to the order of keys in band_defs.
    """
    band_powers = []
    for band_name, (f_min, f_max) in band_defs.items():
        freq_indices = (freqs >= f_min) & (freqs < f_max)

        if not jnp.any(freq_indices):
            avg_power = 0.0
        else:
            # Handle ND arrays by slicing the last dimension (frequencies)
            if psd.ndim > 1:
                band_psd_values = psd[..., freq_indices]
            else:
                band_psd_values = psd[freq_indices]

            # Average over all dimensions (Batch, Cells, Freqs in band)
            avg_power = jnp.mean(band_psd_values)

        band_powers.append(avg_power)

    return band_powers


def net_trainer_temp1(net, optimizer, dataloader, epoch_n=100, initial_params=None, optimal_params=None, training_log=None):

    if initial_params is None:
        initial_params = net.get_parameters()

    if optimal_params is None:
        print("Starting training...")
        opt_params = initial_params
    else:
        print("Continuing training...")
        opt_params = optimal_params

    if training_log is None:
        training_log = {
                "loss": [],
                "gsdr_alpha": [], # Renamed from alpha
                "loss_opt": [],
                "avg_bounded_params": [],
                "std_bounded_params": [],
                "avg_gAMPA": [],
                "std_gAMPA": [],
                "avg_gGABAa": [],
                "std_gGABAa": [],
                "mcdp_factors": [],
                "gamma_band": [],
                "beta_band": [],
                "alpha_band": [],
                "theta_band": []
              }
        pre_cnt = 0

    else:
        pre_cnt = len(training_log["loss"])

    opt_state = optimizer.init(opt_params)
    key = jax.random.PRNGKey(0)

    mid_params = []
    epoch_cnt = 0

    for epoch in range(epoch_n):

        key, step_key = jax.random.split(key)
        epoch_loss_accum = 0.0
        batch_count_for_avg = 0
        current_epoch_had_nan = False

        # Accumulate band powers for the epoch
        epoch_band_powers = [0.0, 0.0, 0.0, 0.0]

        for batch_ind, batch in enumerate(dataloader):
            # print("/")
            current_batch, label_batch = batch
            (loss_val, traces), gradient = jitted_grad(opt_params, current_batch, label_batch)

            # Calculate PSD and Bands for logging
            mean_traces = jnp.mean(traces, axis=1) # (Batch, Time)
            mean_trace_batch = jnp.mean(mean_traces, axis=0) # (Time,)
            freq_t, psd_t = compute_unscaled_psd_from_trace(mean_trace_batch, dt_global)
            band_powers_list = calculate_psd_bands(psd_t, freq_t, band_definitions)
            # print(band_powers_list)
            # print(freq_t)
            # print()

            for i in range(4):
                epoch_band_powers[i] += band_powers_list[i]

            if jnp.isnan(loss_val):
                print(f"Epoch {epoch + pre_cnt}, Batch {batch_ind}: epoch voided due to nan in ODE/PDE. Reverting to last optimal state.")
                current_epoch_had_nan = True
                # GSDR.update_fn handles the reset when value=NaN is passed.
                mcdp_factors = None # No MCDP update on NaN
            else:
                epoch_loss_accum += loss_val
                batch_count_for_avg += 1
                # Calculate MCDP factors from traces
                mcdp_factors = calculate_mcdp(traces)

            current_network_params = transform.forward(opt_params)
            updates, opt_state = optimizer.update(gradient, opt_state,
                params=current_network_params,
                value=loss_val,
                key=step_key,
                mcdp_factors=mcdp_factors) # Pass MCDP factors
            opt_params = optax.apply_updates(opt_params, updates)

        # After all batches in an epoch
        if current_epoch_had_nan:
            epoch_cnt -= 1
            current_avg_loss = jnp.nan
            avg_physical_params = jnp.nan
            std_physical_params = jnp.nan
            avg_gAMPA = jnp.nan
            std_gAMPA = jnp.nan
            avg_gGABAa = jnp.nan
            std_gGABAa = jnp.nan
            avg_band_powers = [jnp.nan]*4
        elif batch_count_for_avg > 0:
            current_avg_loss = epoch_loss_accum / batch_count_for_avg

            # Params Stats
            physical_params = transform.forward(opt_params)
            all_physical_params_flat = []
            gAMPA_vals = []
            gGABAa_vals = []

            for param_group in physical_params:
                for name, param_array in param_group.items():
                    flat = param_array.flatten()
                    all_physical_params_flat.append(flat)
                    if name == 'gAMPA':
                        gAMPA_vals.append(flat)
                    elif name == 'gGABAa':
                        gGABAa_vals.append(flat)

            all_flat = jnp.concatenate(all_physical_params_flat)
            avg_physical_params = jnp.mean(all_flat)
            std_physical_params = jnp.std(all_flat)

            gAMPA_flat = jnp.concatenate(gAMPA_vals) if gAMPA_vals else jnp.array([])
            avg_gAMPA = jnp.mean(gAMPA_flat) if gAMPA_flat.size > 0 else 0.0
            std_gAMPA = jnp.std(gAMPA_flat) if gAMPA_flat.size > 0 else 0.0

            gGABAa_flat = jnp.concatenate(gGABAa_vals) if gGABAa_vals else jnp.array([])
            avg_gGABAa = jnp.mean(gGABAa_flat) if gGABAa_flat.size > 0 else 0.0
            std_gGABAa = jnp.std(gGABAa_flat) if gGABAa_flat.size > 0 else 0.0

            # Average band powers
            avg_band_powers = [p / batch_count_for_avg for p in epoch_band_powers]
        else:
            current_avg_loss = jnp.inf
            avg_physical_params = jnp.nan
            std_physical_params = jnp.nan
            avg_gAMPA = jnp.nan
            std_gAMPA = jnp.nan
            avg_gGABAa = jnp.nan
            std_gGABAa = jnp.nan
            avg_band_powers = [jnp.nan]*4

        if not current_epoch_had_nan:
          print(f"epoch {epoch_cnt}, avg. loss {current_avg_loss:.4f}, alpha {opt_state.a:.4f}, loss_opt {opt_state.loss_opt:.4f}")
          print(f"  Params: Avg={avg_physical_params:.4f}, Std={std_physical_params:.4f}")
          print(f"  gAMPA: Avg={avg_gAMPA:.4f}, Std={std_gAMPA:.4f}")
          print(f"  gGABAa: Avg={avg_gGABAa:.4f}, Std={std_gGABAa:.4f}")

          training_log["loss"].append(current_avg_loss)
          training_log["gsdr_alpha"].append(opt_state.a)
          training_log["loss_opt"].append(opt_state.loss_opt)
          training_log["avg_bounded_params"].append(avg_physical_params)
          training_log["std_bounded_params"].append(std_physical_params)

          training_log["avg_gAMPA"].append(avg_gAMPA)
          training_log["std_gAMPA"].append(std_gAMPA)
          training_log["avg_gGABAa"].append(avg_gGABAa)
          training_log["std_gGABAa"].append(std_gGABAa)

          training_log["mcdp_factors"].append(mcdp_factors)
          training_log["gamma_band"].append(avg_band_powers[0])
          training_log["beta_band"].append(avg_band_powers[1])
          training_log["alpha_band"].append(avg_band_powers[2])
          training_log["theta_band"].append(avg_band_powers[3])

        if epoch_cnt % 10 == 0 and epoch_cnt > 10:
            mid_params.append(transform.forward(opt_params))

        epoch_cnt += 1

    final_params = transform.forward(opt_params)

    return final_params, training_log, mid_params


## Visualizations

In [ ]:
test_net_vis_comp_psd(net, net.get_parameters(), inputs[1], labels[0], labels[1], interval1=(1, 500), interval2=(501, 1000), savename="post_psd_1.svg")
test_net_vis_comp_tfr(net, net.get_parameters(), inputs[1], label_psd=None, interval1=(1, 500), interval2=(501, 1000), savename="post_tfr_1.svg")

In [ ]:
test_net_vis_comp_psd(net, net.get_parameters(), inputs[1]*1.0, labels[0], labels[1], interval1=(1, 500), interval2=(501, 1000), savename="mid_psd_1")
test_net_vis_comp_tfr(net, net.get_parameters(), inputs[1]*1.0, label_psd=None, interval1=(1, 500), interval2=(501, 1000), savename="mid_tfr_1")

## Optimization

### Spectral tuning

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_sgd_dynamic_a000_3"
optimizer_inner = SDR(learning_rate=1e-1, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=3.0,
    a_init=0.0,
    a_dynamic=True,
    lambda_d=1.0,
    checkpoint_n=5,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_sdr_dynamic_a000_3"
optimizer_inner = optax.sgd(learning_rate=1e-1, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=3.0,
    a_init=0.0,
    a_dynamic=True,
    lambda_d=1.0,
    checkpoint_n=5,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_static_a000_3"
optimizer_inner = optax.sgd(learning_rate=1e-1, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=3.0,
    a_init=0.0,
    a_dynamic=True,
    lambda_d=1.0,
    checkpoint_n=5,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_static_a000_3"
optimizer_inner = SDR(learning_rate=1e-1, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=3.0,
    a_init=0.0,
    a_dynamic=False,
    lambda_d=1.0,
    checkpoint_n=6,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_static_a002_3"
optimizer_inner = SDR(learning_rate=1e-2, momentum=0.2)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=3.0,
    a_init=0.2,
    a_dynamic=False,
    lambda_d=1.0,
    checkpoint_n=6,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_static_a004_4"
optimizer_inner = SDR(learning_rate=1.0, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.uniform,
    deselection_threshold=3.0,
    a_init=0.4,
    a_dynamic=False,
    lambda_d=1.0,
    checkpoint_n=5,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_static_a006_3"
optimizer_inner = SDR(learning_rate=1e-1, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=3.0,
    a_init=0.6,
    a_dynamic=False,
    lambda_d=1.0,
    checkpoint_n=5,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_static_a008_3"
optimizer_inner = SDR(learning_rate=1e-1, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=2.0,
    a_init=0.8,
    a_dynamic=False,
    lambda_d=1.0,
    checkpoint_n=5,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_static_a010_2"
optimizer_inner = SDR(learning_rate=1e-1, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=2.0,
    a_init=1.0,
    a_dynamic=False,
    lambda_d=1.0,
    checkpoint_n=5,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_static_a010_3"
optimizer_inner = SDR(learning_rate=1e-1, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=2.0,
    a_init=1.0,
    a_dynamic=False,
    lambda_d=1.0,
    checkpoint_n=5,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_dynamic_a002_3"
optimizer_inner = SDR(learning_rate=1e-1, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=2.0,
    a_init=0.0,
    a_dynamic=True,
    lambda_d=1.0,
    checkpoint_n=6,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

In [ ]:
key = jax.random.PRNGKey(421)
initial_params_0 = net.get_parameters()
model_save_name = "model_dynamic_a003"
optimizer_inner = optax.sgd(learning_rate=1e-1, momentum=0.4)

optimizer = GSDR(
    inner_optimizer=optimizer_inner,
    delta_distribution=jax.random.normal,
    deselection_threshold=2.0,
    a_init=0.0,
    a_dynamic=True,
    lambda_d=1.0,
    checkpoint_n=6,
    mcdp=True
)

final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 500)
# final_params_0, log_net_0, mid_params_0 = net_trainer_temp1(net, optimizer, dataloader, 20, initial_params=final_params_0, optimal_params=final_params_0, training_log=log_net_0)
save_jnn(model_save_name, model_save_path, net, initial_params_0, mid_params_0, final_params_0, log_net_0)

## Evaluation

In [ ]:
t_max_vs = 1500.0
t_onset_vs = 500.0
t_dur_vs = 500.0
single_input_amp = inputs[2]

levels = 2
time_points_vs = t_max_vs / dt_global + 1
checkpoints_vs = [int(np.ceil(time_points_vs**(1/levels))) for _ in range(levels)]

ac_current = noise_current_ac(
    i_delay=t_onset_vs,
    i_dur=t_dur_vs,
    amp_n=0.0,
    amp_b=single_input_amp,
    delta_t=dt_global,
    t_max=t_max_vs,
    spect=jnp.array([120.0])
)

sim_current = ac_current

this_params = final_params_0

net.delete_stimuli()
net.delete_recordings()
num_posts_to_select = int(Ne/2)
posts_pool_indices = jnp.concat([jnp.arange(0, Ne)])
key_conn = jax.random.PRNGKey(5678)
selected_post_indices = np.array(jax.random.choice(key_conn, posts_pool_indices, shape=(num_posts_to_select,), replace=False))

data_stimuli_single_run = net.cell(selected_post_indices).branch(1).loc(0.0).data_stimulate(sim_current) # Changed loc(1.0) to loc(0.0)
net.cell([i for i in range(Ne+Nig+Nil)]).branch(0).loc(0.0).record()

print(f"Running simulation with input amplitude: {single_input_amp:.2f} nA...")
recorded_voltages_pre_training = jx.integrate(net, params=this_params, data_stimuli=data_stimuli_single_run, checkpoint_lengths=checkpoints_vs)
print(f"Simulation completed. Recorded voltages shape: {recorded_voltages_pre_training.shape}")
time_axis = jnp.arange(20, t_max_vs, dt_global)

trace_vis_tfr(recorded_voltages_pre_training)

In [ ]:
test_net_vis_comp_psd(net, final_params_0, inputs[1], labels[0], labels[1], interval1=(1, 500), interval2=(501, 1000), savename="PSD_comp_1.svg")
test_net_vis_comp_tfr(net, final_params_0, inputs[1], label_psd=None, interval1=(1, 500), interval2=(501, 1000), savename="TFR_raster_1.svg")

In [ ]:
import matplotlib.pyplot as plt
import jax.numpy as jnp
import numpy as np
from scipy import ndimage
import os

def plot_model_details(model_name, log_net, init_params, final_params):
    """
    Plots detailed training logs and connectivity matrices for a single model.
    """
    # --- Figure 1: Training Trajectories ---
    fig1, axs = plt.subplots(4, 1, figsize=(12, 16), sharex=True)
    fig1.suptitle(f"Model: {model_name} - Training Dynamics", fontsize=16)

    epochs = np.arange(len(log_net['loss']))

    # 1. Loss
    loss = np.array(log_net['loss'])
    # loss_opt = np.array(log_net['loss_opt'])
    axs[0].plot(epochs, loss, color='r', label='Loss', linewidth=1.5)
    # axs[0].plot(epochs, loss_opt, color='black', linestyle='--', label='Optimal Loss', alpha=0.5)
    axs[0].set_ylabel("Loss")
    axs[0].set_title("Loss per Trial")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)

    # 2. Correlation (MCDP Factors)
    # Extract mean absolute correlation per epoch
    mcdp_corrs = []
    for epoch_factors in log_net['mcdp_factors']:
        if epoch_factors is None:
            mcdp_corrs.append(0)
            continue

        # epoch_factors is typically a list of dicts: [{'gAMPA': ...}, {'gGABAa': ...}]
        # We'll flatten all arrays and take the mean absolute value
        all_factors = []
        for group in epoch_factors:
            for val in group.values():
                all_factors.append(np.array(val).flatten())

        if all_factors:
            flat_factors = np.concatenate(all_factors)
            mcdp_corrs.append(np.mean(np.abs(flat_factors)))
        else:
            mcdp_corrs.append(0)

    axs[1].plot(epochs, mcdp_corrs, color='purple', label='Mean |Correlation|')
    axs[1].set_ylabel("Correlation strength")
    axs[1].set_title("Correlation to Data (MCDP Factors)")
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)

    # 3. Band Powers
    if 'gamma_band' in log_net:
        axs[2].plot(epochs, log_net['gamma_band'], color='green', linestyle='-', label='Gamma (35-60Hz)')
        axs[2].plot(epochs, log_net['beta_band'], color='orange', linestyle='--', label='Beta (13-30Hz)')
        axs[2].plot(epochs, log_net['alpha_band'], color='blue', linestyle='--', label='Alpha (8-12Hz)')
        # axs[2].plot(epochs, log_net['theta_band'], color='grey', linestyle=':', label='Theta (4-8Hz)')
    axs[2].set_ylabel("Relative Power")
    axs[2].set_title("Band Powers per Trial")
    axs[2].legend()
    axs[2].grid(True, alpha=0.3)

    # 4. Parameters (Proxy for Excitability/FR)
    # Plotting avg gAMPA and gGABAa as we don't have per-trial FR logs
    if 'avg_gAMPA' in log_net:
        axs[3].plot(epochs, log_net['avg_gAMPA'], color='red', label='Avg gAMPA')
        axs[3].plot(epochs, log_net['avg_gGABAa'], color='blue', label='Avg gGABAa')
    axs[3].set_ylabel("Conductance (uS)")
    axs[3].set_xlabel("Trial (Epoch)")
    axs[3].set_title("Mean Synaptic Weights (Excitability Proxy)")
    axs[3].legend()
    axs[3].grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.show()

    # --- Figure 2: Connectivity & MCDP Matrices ---
    fig2, axs2 = plt.subplots(2, 2, figsize=(14, 12))
    fig2.suptitle(f"Model: {model_name} - Connectivity & MCDP (First 50 Neurons)", fontsize=16)

    def get_dense_matrix(params, subset_n=50):
        # Initialize dense matrix
        N_total = Ne + Nig + Nil
        W = np.zeros((N_total, N_total))

        # Fill AMPA
        # We need to access the parameter arrays from the 'params' dictionary structure
        # The structure is usually a list of dicts, e.g. [{'gAMPA': ...}, {'gGABAa': ...}]
        # We assume order AMPA then GABA based on 'transform' definition in training

        # Helper to find param by name
        def get_param(p_list, name):
            for group in p_list:
                if name in group:
                    return np.array(group[name])
            return None

        g_ampa = get_param(params, 'gAMPA')
        g_gaba = get_param(params, 'gGABAa')

        if g_ampa is not None and len(g_ampa) == len(ampa_pre_inds):
            W[ampa_pre_inds, ampa_post_inds] += g_ampa.flatten()

        if g_gaba is not None and len(g_gaba) == len(gaba_pre_inds):
            W[gaba_pre_inds, gaba_post_inds] -= g_gaba.flatten() # Negative for inhibition visualization

        return W[:subset_n, :subset_n]

    def get_mcdp_matrix(mcdp_list, subset_n=50):
        N_total = Ne + Nig + Nil
        M = np.zeros((N_total, N_total))

        if mcdp_list is None: return M

        # mcdp_list is [{'gAMPA': ...}, {'gGABAa': ...}]
        # Helper
        def get_val(p_list, name):
            for group in p_list:
                if name in group:
                    return np.array(group[name])
            return None

        r_ampa = get_val(mcdp_list, 'gAMPA')
        r_gaba = get_val(mcdp_list, 'gGABAa')

        if r_ampa is not None and len(r_ampa) == len(ampa_pre_inds):
            M[ampa_pre_inds, ampa_post_inds] += r_ampa.flatten()

        if r_gaba is not None and len(r_gaba) == len(gaba_pre_inds):
            M[gaba_pre_inds, gaba_post_inds] += r_gaba.flatten()

        return M[:subset_n, :subset_n]

    # 1. Initial Connectivity
    W_init = get_dense_matrix(init_params)
    im1 = axs2[0, 0].imshow(W_init, cmap='bwr', vmin=-10, vmax=10)
    axs2[0, 0].set_title("Initial Connectivity (Weights)")
    plt.colorbar(im1, ax=axs2[0, 0])

    # 2. Final Connectivity
    W_final = get_dense_matrix(final_params)
    im2 = axs2[0, 1].imshow(W_final, cmap='bwr', vmin=-10, vmax=10)
    axs2[0, 1].set_title("Final Connectivity (Weights)")
    plt.colorbar(im2, ax=axs2[0, 1])

    # 3. Initial MCDP
    # Use the first available non-None mcdp factors
    first_mcdp = next((x for x in log_net['mcdp_factors'] if x is not None), None)
    M_init = get_mcdp_matrix(first_mcdp)
    im3 = axs2[1, 0].imshow(M_init, cmap='viridis', vmin=-2, vmax=2)
    axs2[1, 0].set_title("Initial MCDP Factors")
    plt.colorbar(im3, ax=axs2[1, 0])

    # 4. Final MCDP
    last_mcdp = log_net['mcdp_factors'][-1] if log_net['mcdp_factors'] else None
    M_final = get_mcdp_matrix(last_mcdp)
    im4 = axs2[1, 1].imshow(M_final, cmap='viridis', vmin=-2, vmax=2)
    axs2[1, 1].set_title("Final MCDP Factors")
    plt.colorbar(im4, ax=axs2[1, 1])

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.show()



## Visualization II

### Load session logs

In [ ]:
loaded_nets = []
loaded_initial_params = []
loaded_mid_params = []
loaded_final_params = []
loaded_log_nets = []
loaded_log_names = []
cnt = 0

# Sort files for consistent ordering
files = sorted([f for f in os.listdir(model_save_path) if f.startswith("refn_")])

for file in files:
    try:
        loaded_net, loaded_initial_param, loaded_mid_param, loaded_final_param, loaded_log_net = load_jnn(file, model_save_path)
        loaded_nets.append(loaded_net)
        loaded_initial_params.append(loaded_initial_param)
        loaded_mid_params.append(loaded_mid_param)
        loaded_final_params.append(loaded_final_param)
        loaded_log_nets.append(loaded_log_net)
        loaded_log_names.append(file)
        # print(f"Loaded {file}")
    except Exception as e:
        print(f"Error loading {file}: {e}")

In [ ]:
def plot_trajectory_bandpowers(log_nets, save_fname=None, band1=None, band2=None, lim1=None, lim2=None, ksw=1):

    if band1 is None:
        band1 = 'gamma_band'
    if band2 is None:
        band2 = 'beta_band'

    if lim1 is None:
        lim1 = (0, 2)
    if lim2 is None:
        lim2 = (0, 2)

    fig, axes = plt.subplots(1, 1, figsize=(10, 10))

    for i, log_net in enumerate(log_nets):

        if 'gsdr_alpha' in log_net:
            alph_t = log_net['gsdr_alpha'][0]
            alpha_key = 'gsdr_alpha'
        elif 'alpha' in log_net:
            alph_t = log_net['alpha'][0]
            alpha_key = 'alpha'
        else:
            alph_t = 0.0
            alpha_key = None

        label = loaded_log_names[i]


        if band1 in log_net and len(log_net[band1]) > 0:

            a_params = jnp.array(log_net[band1])
            b_params = jnp.array(log_net[band2])

            if len(a_params) > ksw:
                a_params = ndimage.uniform_filter1d(a_params, ksw)
                b_params = ndimage.uniform_filter1d(b_params, ksw)

            if a_params[0] != 0:
                a_params = a_params / a_params[0]
            if b_params[0] != 0:
                b_params = b_params / b_params[0]

            color = plt.colormaps['turbo'](i / len(log_nets))

            for j in range(len(a_params) - 1):
                axes.annotate(
                    '', xy=(a_params[j+1], b_params[j+1]), xytext=(a_params[j], b_params[j]),
                    arrowprops=dict(arrowstyle='->', color=color, linewidth=0.5)
                )

            axes.plot(a_params[0], b_params[0], marker='x', markersize=10, color=(0, 0, 0), label=None)
            axes.plot(a_params[-1], b_params[-1], marker='o', markersize=10, color=color, label=f'{i}: {label}')
            axes.text(a_params[-1], b_params[-1], str(i), fontsize=12, fontweight='bold', color='black')

    axes.set_title('RelParamsChange')
    axes.set_xlabel(band1)
    axes.set_ylabel(band2)

    # axes.plot(jnp.linspace(0, 30, 10), jnp.linspace(0, 30, 10), linestyle="--", label=None, color=(0, 0, 0))
    axes.set_xlim(lim1)
    axes.set_ylim(lim2)

    axes.legend(loc='lower left')
    axes.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    if save_fname is not None:
        plt.savefig(save_fname, format='svg')

    plt.show()


def plot_trajectory_ratio(log_nets, save_fname=None, band11=None, band12=None, band21=None, band22=None, lim1=None, lim2=None, ksw=1):

    if lim1 is None:
        lim1 = (0, 2)
    if lim2 is None:
        lim2 = (0, 2)

    fig, axes = plt.subplots(1, 1, figsize=(10, 10))

    for i, log_net in enumerate(log_nets):

        if 'gsdr_alpha' in log_net:
            alph_t = log_net['gsdr_alpha'][0]
            alpha_key = 'gsdr_alpha'
        elif 'alpha' in log_net:
            alph_t = log_net['alpha'][0]
            alpha_key = 'alpha'
        else:
            alph_t = 0.0
            alpha_key = None

        label = loaded_log_names[i]

        if band11 in log_net and len(log_net[band11]) > 0:

            a_params = jnp.array(log_net[band11])
            b_params = jnp.array(log_net[band12])
            c_params = jnp.array(log_net[band21])
            d_params = jnp.array(log_net[band22])

            if len(a_params) > ksw:
                a_params = ndimage.uniform_filter1d(a_params, ksw)
                b_params = ndimage.uniform_filter1d(b_params, ksw)
                c_params = ndimage.uniform_filter1d(c_params, ksw)
                d_params = ndimage.uniform_filter1d(d_params, ksw)

            # Calculate Ratios element-wise
            # Add epsilon to denominator to prevent division by zero
            ratio_x = a_params / (b_params + 1e-9)
            ratio_y = c_params / (d_params + 1e-9)

            # Normalize Ratios to start at 1.0
            if ratio_x[0] != 0:
                ratio_x = ratio_x / ratio_x[0]
            if ratio_y[0] != 0:
                ratio_y = ratio_y / ratio_y[0]

            color = plt.colormaps['turbo'](i / len(log_nets))

            for j in range(len(ratio_x) - 1):
                axes.annotate(
                    '', xy=(ratio_x[j+1], ratio_y[j+1]), xytext=(ratio_x[j], ratio_y[j]),
                    arrowprops=dict(arrowstyle='->', color=color, linewidth=0.5)
                )

            axes.plot(ratio_x[0], ratio_y[0], marker='x', markersize=10, color=(0, 0, 0), label=None)
            axes.plot(ratio_x[-1], ratio_y[-1], marker='o', markersize=10, color=color, label=f'{i}: {label}')
            axes.text(ratio_x[-1], ratio_y[-1], str(i), fontsize=12, fontweight='bold', color='black')

    axes.set_title('Relative Parameter Ratio Change')
    axes.set_xlabel(f"{band11} / {band12} (Normalized)")
    axes.set_ylabel(f"{band21} / {band22} (Normalized)")

    axes.set_xlim(lim1)
    axes.set_ylim(lim2)

    axes.legend(loc='lower right')
    axes.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    if save_fname is not None:
        plt.savefig(save_fname, format='svg')

    plt.show()

### Log figures

In [ ]:
def plot_training_logs(log_nets, save_fname=None):

    fig, axes = plt.subplots(4, 1, figsize=(20, 32))

    final_stats = [] # Store data for post-loop annotation
    model_optimals = [] # Store (name, optimal_loss)

    for i, log_net in enumerate(log_nets):

        if 'gsdr_alpha' in log_net:
            alph_t = log_net['gsdr_alpha'][0]
            alpha_key = 'gsdr_alpha'
        elif 'alpha' in log_net:
            alph_t = log_net['alpha'][0]
            alpha_key = 'alpha'
        else:
            alph_t = 0.0
            alpha_key = None

        if alpha_key and len(log_net[alpha_key]) > 10 and alph_t != log_net[alpha_key][10]:
             label = f"{i}: {loaded_log_names[i]}"
        else:
             label = f"{i}: Static Alph-{alph_t}"

        epochs = jnp.arange(len(log_net['loss']))
        loss_data = jnp.array(log_net['loss'])
        loss_data = jnp.where(jnp.isfinite(loss_data), loss_data, 1e-6) # Replace inf/nan
        loss_t = loss_data

        smoothing_window = 5
        color = plt.colormaps['turbo'](i / len(log_nets))

        # Plot 1: Instantaneous Loss
        if len(loss_t) > smoothing_window:
            loss_tm = ndimage.uniform_filter1d(loss_t, size=smoothing_window)
            std_loss_t = ndimage.uniform_filter1d(jnp.abs(loss_t - loss_tm), size=smoothing_window*3)

            axes[0].plot(epochs, loss_tm, label=label, color=color)
            axes[0].fill_between(epochs, loss_tm - std_loss_t, loss_tm + std_loss_t, alpha=0.1, color=color)
        else:
             axes[0].plot(epochs, loss_t, label=label, color=color)

        axes[0].set_yscale('log')
        axes[0].set_title('Loss per Trial')
        axes[0].set_ylabel('Loss')
        axes[0].set_yticks([10000, 5000, 2500, 1000, 750, 500])
        axes[0].get_yaxis().set_major_formatter(plt.ScalarFormatter())
        axes[0].set_ylim(500, 7500)

        # Plot 2: Optimal Loss up to Trial
        cum_min_loss = np.minimum.accumulate(loss_data)
        axes[1].plot(epochs, cum_min_loss, label=label, color=color, linestyle='-', linewidth=4.0)

        if len(cum_min_loss) > 0:
            final_val = cum_min_loss[-1]
            final_stats.append({
                'val': final_val,
                'color': color,
                'id': i,
                'x': epochs[-1]
            })

            # Store for grouping
            last_100_avg = np.mean(loss_data[-100:]) if len(loss_data) >= 100 else np.mean(loss_data)
            model_optimals.append({
                'name': loaded_log_names[i],
                'optimal_loss': float(final_val),
                'last100_loss': float(last_100_avg),
                'color': color,
                'id': i
            })

        axes[1].set_yscale('log')
        axes[1].set_title('Optimal Loss per Trial (Log Scale)')
        axes[1].set_ylabel('Min Loss')
        axes[1].set_yticks([10000, 5000, 2500, 1000, 750, 500])
        axes[1].get_yaxis().set_major_formatter(plt.ScalarFormatter())
        axes[1].set_ylim(500, 7500)

    # Sort stats and place evenly spaced text for Subplot 2
    if final_stats:
        final_stats.sort(key=lambda x: x['val'])
        y_min_log = np.log10(500)
        y_max_log = np.log10(7500)
        log_positions = np.linspace(y_min_log, y_max_log, len(final_stats) + 2)[1:-1]
        y_positions = 10**log_positions

        for idx, item in enumerate(final_stats):
            axes[1].text(1.02, y_positions[idx], f"{item['id']}. {item['val']:.0f}",
                         color=item['color'], fontsize=12, fontweight='bold',
                         va='center', transform=axes[1].get_yaxis_transform())

    # --- Group Statistics Logic ---
    groups = {
        "SDR": [],
        "SGD": [],
        "Dynamic SSa": [],
        "Moderate SSa": [],
        "High SSa": [],
        "All": [],
        "No SSa": []
    }

    for m in model_optimals:
        name = m['name']
        if "model_sdr" in name: groups["SDR"].append(m)
        if "model_sgd" in name: groups["SGD"].append(m)
        if "dynamic" in name: groups["Dynamic SSa"].append(m)
        if any(x in name for x in ["static_a002", "static_a004", "static_a006"]): groups["Moderate SSa"].append(m)
        if any(x in name for x in ["static_a008", "static_a010"]): groups["High SSa"].append(m)
        groups["All"].append(m)
        if "a000" in name and "dynamic" not in name: groups["No SSa"].append(m)

    group_names = list(groups.keys())

    # --- Subplot 3: Group Statistics (Optimal Loss) ---
    group_means_opt = []
    group_stds_opt = []
    group_mins_opt = []
    group_maxs_opt = []
    group_data_points_opt = []

    for g_name in group_names:
        items = groups[g_name]
        if not items:
            group_means_opt.append(np.nan)
            group_stds_opt.append(np.nan)
            group_mins_opt.append(np.nan)
            group_maxs_opt.append(np.nan)
            group_data_points_opt.append([])
            continue

        vals = [item['optimal_loss'] for item in items]
        group_means_opt.append(np.mean(vals))
        group_stds_opt.append(np.std(vals))
        group_mins_opt.append(np.min(vals))
        group_maxs_opt.append(np.max(vals))
        group_data_points_opt.append(items)

    x_pos = np.arange(len(group_names))
    means_opt = np.array(group_means_opt)
    stds_opt = np.array(group_stds_opt)
    mins_opt = np.array(group_mins_opt)
    maxs_opt = np.array(group_maxs_opt)
    yerr_range_opt = np.array([means_opt - mins_opt, maxs_opt - means_opt])

    rect_width = 0.3
    axes[2].errorbar(x_pos, means_opt, yerr=yerr_range_opt, fmt='none',
                     ecolor='black', elinewidth=3, capsize=5, label='Min / Max', zorder=1)
    axes[2].bar(x_pos, height=stds_opt*2, bottom=means_opt-stds_opt, width=rect_width,
                color='lightgray', edgecolor='black', alpha=0.5, label='Mean ± 1 STD', zorder=2)
    axes[2].scatter(x_pos, means_opt, s=100, marker='D', color='black', zorder=5, label='Group Mean')

    for idx, items in enumerate(group_data_points_opt):
        if not items: continue
        ys = [it['optimal_loss'] for it in items]
        colors = [it['color'] for it in items]
        xs = np.random.normal(idx, 0.05, size=len(ys))
        axes[2].scatter(xs, ys, c=colors, s=50, alpha=0.8, zorder=4, edgecolors='black')

    axes[2].set_title('Group Analysis: Optimal Loss')
    axes[2].set_xticks(x_pos)
    axes[2].set_xticklabels(group_names, rotation=15, ha='right', fontsize=12)
    axes[2].set_ylabel('Optimal Loss')
    axes[2].set_yscale('log')
    axes[2].grid(True, axis='y', linestyle='--', alpha=0.6)

    # --- Subplot 4: Group Statistics (Last 100 Avg Loss) ---
    group_means_l100 = []
    group_stds_l100 = []
    group_mins_l100 = []
    group_maxs_l100 = []

    for g_name in group_names:
        items = groups[g_name]
        if not items:
            group_means_l100.append(np.nan)
            group_stds_l100.append(np.nan)
            group_mins_l100.append(np.nan)
            group_maxs_l100.append(np.nan)
            continue

        vals = [item['last100_loss'] for item in items]
        group_means_l100.append(np.mean(vals))
        group_stds_l100.append(np.std(vals))
        group_mins_l100.append(np.min(vals))
        group_maxs_l100.append(np.max(vals))

    means_l100 = np.array(group_means_l100)
    stds_l100 = np.array(group_stds_l100)
    mins_l100 = np.array(group_mins_l100)
    maxs_l100 = np.array(group_maxs_l100)
    yerr_range_l100 = np.array([means_l100 - mins_l100, maxs_l100 - means_l100])

    axes[3].errorbar(x_pos, means_l100, yerr=yerr_range_l100, fmt='none',
                     ecolor='black', elinewidth=3, capsize=5, label='Min / Max', zorder=1)
    axes[3].bar(x_pos, height=stds_l100*2, bottom=means_l100-stds_l100, width=rect_width,
                color='lightblue', edgecolor='black', alpha=0.5, label='Mean ± 1 STD', zorder=2)
    axes[3].scatter(x_pos, means_l100, s=100, marker='D', color='black', zorder=5, label='Group Mean')

    for idx, items in enumerate(group_data_points_opt):
        if not items: continue
        ys = [it['last100_loss'] for it in items]
        colors = [it['color'] for it in items]
        xs = np.random.normal(idx, 0.05, size=len(ys))
        axes[3].scatter(xs, ys, c=colors, s=50, alpha=0.8, zorder=4, edgecolors='black')

    axes[3].set_title('Group Analysis: Avg Loss (Last 100 Trials)')
    axes[3].set_xticks(x_pos)
    axes[3].set_xticklabels(group_names, rotation=15, ha='right', fontsize=12)
    axes[3].set_ylabel('Avg Loss')
    axes[3].set_yscale('log')
    axes[3].grid(True, axis='y', linestyle='--', alpha=0.6)

    for ax in axes[:2]:
        ax.legend(loc='upper right')
        ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    if save_fname is not None:
        plt.savefig(save_fname, format='svg')

    plt.show()

plot_training_logs(loaded_log_nets, "GSDR_alpha_logs_rw.svg")

In [ ]:
def plot_trajectory_3d(log_nets, save_fname=None, band1=None, band2=None, band3=None,
                       lim1=None, lim2=None, lim3=None, ksw=1):

    if lim1 is None: lim1 = (0, 2)
    if lim2 is None: lim2 = (0, 2)
    if lim3 is None: lim3 = (0, 2)

    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(111, projection='3d')

    for i, log_net in enumerate(log_nets):

        if 'gsdr_alpha' in log_net:
            alph_t = log_net['gsdr_alpha'][0]
            alpha_key = 'gsdr_alpha'
        elif 'alpha' in log_net:
            alph_t = log_net['alpha'][0]
            alpha_key = 'alpha'
        else:
            alph_t = 0.0
            alpha_key = None

        label = loaded_log_names[i]

        if band1 in log_net and len(log_net[band1]) > 0:

            a_params = jnp.array(log_net[band1])
            b_params = jnp.array(log_net[band2])
            c_params = jnp.array(log_net[band3])

            if len(a_params) > ksw:
                a_params = ndimage.uniform_filter1d(a_params, ksw)
                b_params = ndimage.uniform_filter1d(b_params, ksw)
                c_params = ndimage.uniform_filter1d(c_params, ksw)

            if a_params[0] != 0: a_params = a_params / a_params[0]
            if b_params[0] != 0: b_params = b_params / b_params[0]
            if c_params[0] != 0: c_params = c_params / c_params[0]

            color = plt.colormaps['turbo'](i / len(log_nets))

            # Plot trajectory line
            ax.plot(a_params, b_params, c_params, color=color, label=f'{i}: {label}')

            # Add arrows using quiver
            # Vectors between consecutive points
            u = a_params[1:] - a_params[:-1]
            v = b_params[1:] - b_params[:-1]
            w = c_params[1:] - c_params[:-1]

            # Plot arrows at the starting point of each segment
            ax.quiver(a_params[:-1], b_params[:-1], c_params[:-1], u, v, w,
                      color=color, arrow_length_ratio=0.1, linewidth=0.8)

            # Start marker
            ax.scatter(a_params[0], b_params[0], c_params[0], marker='x', s=50, color='black')

            # End marker
            ax.scatter(a_params[-1], b_params[-1], c_params[-1], marker='o', s=50, color=color)

            # End label
            ax.text(a_params[-1], b_params[-1], c_params[-1], str(i),
                    fontsize=12, fontweight='bold', color='black')

    ax.set_title('RelParamsChange 3D')
    ax.set_xlabel(band1)
    ax.set_ylabel(band2)
    ax.set_zlabel(band3)

    ax.set_xlim(lim1)
    ax.set_ylim(lim2)
    ax.set_zlim(lim3)

    ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1), fontsize='small')
    ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    if save_fname is not None:
        plt.savefig(save_fname, format='svg')

    plt.show()

plot_trajectory_3d(loaded_log_nets, "GSDR_trj_params_3d.svg",
                   "std_bounded_params", "avg_bounded_params", "avg_gAMPA",
                   lim1=(0.8, 3.2), lim2=(0.6, 1.2), lim3=(0.5, 1.5), ksw=3)

In [ ]:
all_loaded_data = []

for file_name in os.listdir(save_dir):
    if file_name.startswith('simulation_logs_') and file_name.endswith('.pkl'):
        full_file_path = os.path.join(save_dir, file_name)
        try:
            # load_logs returns voltages, indices, losses
            voltages, indices, losses = load_logs(full_file_path)
            all_loaded_data.append({
                'file_name': file_name,
                'log_loss': losses,
                'log_voltages_ix': indices,
                'log_voltages': voltages # Store voltages if needed for future subtasks
            })
        except Exception as e:
            print(f"Error loading {file_name}: {e}")

print(f"Loaded {len(all_loaded_data)} simulation log files.")

In [ ]:
plot_trajectory_ratio(loaded_log_nets, "GSDR_trj_params2.svg", "std_gAMPA", "avg_gAMPA", "std_bounded_params", "avg_bounded_params",
               lim1=(0.0, 8.0), lim2=(0.0, 3.0), ksw=5)


In [ ]:
plot_trajectory_bandpowers(loaded_log_nets, "GSDR_trj_params_2d.svg",
                   "std_bounded_params", "avg_bounded_params",
                   lim1=(0.8, 3.2), lim2=(0.6, 1.2), ksw=5)

In [ ]:
plot_trajectory_bandpowers(loaded_log_nets, "GSDR_trj_params1.svg", "std_bounded_params", "avg_bounded_params",
                           lim1=(0.8, 3.2), lim2=(0.6, 1.2), ksw=5)

## Network augmentation

In [ ]:
def plot_connectivity_grid(loaded_params, log_names, net_template):
    """
    Plots a grid of connectivity matrices for multiple models.
    Assumes params contain 'gAMPA' and 'gGABAa'.
    """
    n_models = len(loaded_params)
    cols = 4
    rows = int(np.ceil(n_models / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = axes.flatten()

    # Get indices from template network
    # We assume all loaded networks share the same architecture (net_eig with fixed seed)
    # Re-extracting just to be safe and self-contained
    view_ampa = net_template.GradedAMPA.edge("all")
    view_gaba = net_template.GradedGABAa.edge("all")

    def get_inds(view):
        if 'pre_cell_index' in view.edges.columns:
            return view.edges['pre_cell_index'].values.astype(int), view.edges['post_cell_index'].values.astype(int)
        else:
            # Fallback for older jaxley versions or different column names
            # Assuming 'pre_index' maps to cell index if 1 compartment per cell or handled externally
            # For this specific notebook, we used node_to_cell mapping in previous cells.
            # Let's use the global mapping logic if available or re-derive.
            # Re-deriving based on node index assuming simple mapping if global_cell_index missing
            if 'global_cell_index' in net_template.nodes.columns:
                node_map = net_template.nodes['global_cell_index'].values
                pre = node_map[view.edges['pre_index'].values.astype(int)]
                post = node_map[view.edges['post_index'].values.astype(int)]
                return pre, post
            return view.edges['pre_index'].values.astype(int), view.edges['post_index'].values.astype(int)

    ampa_pre, ampa_post = get_inds(view_ampa)
    gaba_pre, gaba_post = get_inds(view_gaba)

    N_total = len(list(net_template.cells))

    for i in range(n_models):
        ax = axes[i]
        params = loaded_params[i]
        name = log_names[i]

        # Build Matrix
        W = np.zeros((N_total, N_total))

        # Helper to find value in list of dicts
        def get_val(p_list, key):
            for group in p_list:
                if key in group:
                    return np.array(group[key]).flatten()
            return np.array([])

        g_ampa = get_val(params, 'gAMPA')
        g_gaba = get_val(params, 'gGABAa')

        if len(g_ampa) == len(ampa_pre):
            W[ampa_pre, ampa_post] += g_ampa

        if len(g_gaba) == len(gaba_pre):
            W[gaba_pre, gaba_post] -= g_gaba # Inhibitory as negative

        # Plot
        im = ax.imshow(W, cmap='bwr', vmin=-10, vmax=10, aspect='auto', interpolation='nearest')
        ax.set_title(f"{i}: {name}", fontsize=8)
        ax.axis('off')

    # Hide unused subplots
    for i in range(n_models, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.colorbar(im, ax=axes.ravel().tolist(), label='Weight (AMPA - GABA)')
    plt.show()

# plot_connectivity_grid(loaded_final_params, loaded_log_names, net)

In [ ]:
import jax.numpy as jnp
import numpy as np
from scipy import ndimage

def _get_matrix_from_params_robust(net_structure, param_name, params_pytree):
    """
    Extracts a dense connectivity matrix for a specific param_name from a JAX PyTree.
    Returns None if parameter is not found in params_pytree.
    Robust to list-of-dicts structure and invalid edges.
    """
    N_cells = len(list(net_structure.cells))

    param_values_flat = None
    # Find values in the list of dicts
    for group in params_pytree:
        if param_name in group:
            param_values_flat = np.array(group[param_name])
            break
        # Check nested values just in case
        for val in group.values():
            if isinstance(val, dict) and param_name in val:
                param_values_flat = np.array(val[param_name])
                break
        if param_values_flat is not None:
            break

    if param_values_flat is None:
        print(f"Warning: Param '{param_name}' not found in source params. Skipping.")
        return None

    print(f"DEBUG: Found source {param_name} with shape {param_values_flat.shape}")

    mat = np.zeros((N_cells, N_cells))

    # Identify synapse type for edge filtering
    synapse_type = None
    if 'AMPA' in param_name: synapse_type = 'GradedAMPA'
    elif 'GABAa' in param_name: synapse_type = 'GradedGABAa'

    if synapse_type and 'type' in net_structure.edges.columns:
        edge_mask = net_structure.edges['type'] == synapse_type
        relevant_edges = net_structure.edges[edge_mask]

        print(f"DEBUG: {param_name} corresponds to {len(relevant_edges)} edges in net_structure")

        if len(relevant_edges) > 0:
            pre_nodes = relevant_edges['pre_index'].values.astype(int)
            post_nodes = relevant_edges['post_index'].values.astype(int)

            # Check for mapping
            if 'global_cell_index' in net_structure.nodes.columns:
                node_to_cell = net_structure.nodes['global_cell_index'].values
            else:
                node_to_cell = net_structure.nodes.index.values

            # Filter invalid nodes
            num_nodes = len(node_to_cell)
            valid_mask = (pre_nodes < num_nodes) & (post_nodes < num_nodes)

            if not np.all(valid_mask):
                print(f"Warning: Found {np.sum(~valid_mask)} invalid edges for param {param_name} in source (out of bounds). Ignoring them.")
                pre_nodes = pre_nodes[valid_mask]
                post_nodes = post_nodes[valid_mask]
                # Also filter values if they correspond 1-to-1
                if len(param_values_flat) == len(relevant_edges):
                    param_values_flat = param_values_flat[valid_mask]

            pre_cells = node_to_cell[pre_nodes]
            post_cells = node_to_cell[post_nodes]

            # Map flat values to matrix
            n_vals = min(len(pre_cells), len(param_values_flat))
            mat[pre_cells[:n_vals], post_cells[:n_vals]] = param_values_flat[:n_vals]

    return mat

def _matrix_to_flat_params_robust(net_structure, param_name, matrix_values):
    """
    Samples values from the upsampled matrix for the target network's edges.
    Robust to invalid edges in target network.
    """
    synapse_type = None
    if 'AMPA' in param_name: synapse_type = 'GradedAMPA'
    elif 'GABAa' in param_name: synapse_type = 'GradedGABAa'

    vals = jnp.array([])

    if synapse_type and 'type' in net_structure.edges.columns:
        edge_mask = net_structure.edges['type'] == synapse_type
        relevant_edges = net_structure.edges[edge_mask]
        num_edges = len(relevant_edges)

        print(f"DEBUG: Target net has {num_edges} edges for {param_name}")

        if num_edges > 0:
            pre_nodes = relevant_edges['pre_index'].values.astype(int)
            post_nodes = relevant_edges['post_index'].values.astype(int)

            if 'global_cell_index' in net_structure.nodes.columns:
                node_to_cell = net_structure.nodes['global_cell_index'].values
            else:
                node_to_cell = net_structure.nodes.index.values

            num_nodes = len(node_to_cell)
            print(f"DEBUG: Target num_nodes={num_nodes}, Max pre={pre_nodes.max()}, Max post={post_nodes.max()}")

            # Prepare result array with zeros
            result_vals = np.zeros(num_edges, dtype=np.float32)

            # Identify valid edges
            valid_mask = (pre_nodes < num_nodes) & (post_nodes < num_nodes)

            if not np.all(valid_mask):
                print(f"Warning: Target net has {np.sum(~valid_mask)} invalid edges for {param_name}. Filling them with 0.")

            # Process valid edges only
            valid_pre_nodes = pre_nodes[valid_mask]
            valid_post_nodes = post_nodes[valid_mask]

            pre_cells = node_to_cell[valid_pre_nodes]
            post_cells = node_to_cell[valid_post_nodes]

            N_mat = matrix_values.shape[0]
            pre_cells = np.clip(pre_cells, 0, N_mat - 1)
            post_cells = np.clip(post_cells, 0, N_mat - 1)

            mapped_vals = matrix_values[pre_cells, post_cells]

            # Assign back to the correct positions
            result_vals[valid_mask] = mapped_vals

            vals = jnp.array(result_vals)

    return vals

def params_augment_robust(source_params, net1_template, net2_target, sigma=2.0):
    N1 = len(list(net1_template.cells))
    N2 = len(list(net2_target.cells))
    print(f"DEBUG: Augmenting params from N1={N1} to N2={N2}")

    zoom_fac = N2 / N1 if N1 > 0 else 1.0

    # 1. Extract matrices from source
    # Returns None if param not found in source_params
    w1_ampa = _get_matrix_from_params_robust(net1_template, 'gAMPA', source_params)
    w1_gaba = _get_matrix_from_params_robust(net1_template, 'gGABAa', source_params)

    flat_ampa = None
    flat_gaba = None

    # 2. Upsample and Smooth only if present
    if w1_ampa is not None:
        print(f"DEBUG: Extracted W1 AMPA shape: {w1_ampa.shape}")
        w2_ampa = ndimage.zoom(w1_ampa, zoom_fac, order=1)
        if w2_ampa.shape[0] != N2:
            temp = np.zeros((N2, N2))
            d = min(N2, w2_ampa.shape[0])
            temp[:d, :d] = w2_ampa[:d, :d]
            w2_ampa = temp
        w2_ampa = ndimage.gaussian_filter(w2_ampa, sigma)
        print(f"DEBUG: Upsampled W2 AMPA shape: {w2_ampa.shape}")
        flat_ampa = _matrix_to_flat_params_robust(net2_target, 'gAMPA', w2_ampa)
        print(f"DEBUG: Generated Flat AMPA for net2 shape: {flat_ampa.shape}")

    if w1_gaba is not None:
        print(f"DEBUG: Extracted W1 GABA shape: {w1_gaba.shape}")
        w2_gaba = ndimage.zoom(w1_gaba, zoom_fac, order=1)
        if w2_gaba.shape[0] != N2:
            temp = np.zeros((N2, N2))
            d = min(N2, w2_gaba.shape[0])
            temp[:d, :d] = w2_gaba[:d, :d]
            w2_gaba = temp
        w2_gaba = ndimage.gaussian_filter(w2_gaba, sigma)
        print(f"DEBUG: Upsampled W2 GABA shape: {w2_gaba.shape}")
        flat_gaba = _matrix_to_flat_params_robust(net2_target, 'gGABAa', w2_gaba)
        print(f"DEBUG: Generated Flat GABA for net2 shape: {flat_gaba.shape}")

    # 3. Inject into target structure
    new_params = net2_target.get_parameters()

    # Update the lists
    updated_params = []
    for group in new_params:
        new_group = group.copy()
        # Check if this group holds the synaptic parameters we want to update
        if 'gAMPA' in new_group and flat_ampa is not None and len(flat_ampa) > 0:
            if len(new_group['gAMPA']) == len(flat_ampa):
                new_group['gAMPA'] = flat_ampa
            else:
                print(f"Warning: Size mismatch for gAMPA. Net2 expects {len(new_group['gAMPA'])}, got {len(flat_ampa)}. Keeping original.")

        if 'gGABAa' in new_group and flat_gaba is not None and len(flat_gaba) > 0:
            if len(new_group['gGABAa']) == len(flat_gaba):
                new_group['gGABAa'] = flat_gaba
            else:
                print(f"Warning: Size mismatch for gGABAa. Net2 expects {len(new_group['gGABAa'])}, got {len(flat_gaba)}. Keeping original.")

        updated_params.append(new_group)

    return updated_params

Ne2 = 36
Nig2 = 8
Nil2 = 6

net2 = net_eig(Ne2, Nig2, Nil2)
# Ensure params are trainable to match structure if needed, or get default params
net2.GradedAMPA.edge("all").make_trainable("gAMPA")
net2.GradedGABAa.edge("all").make_trainable("gGABAa")

net1 = loaded_nets[7]
params1 = loaded_initial_params[7]

# Call params_augment_robust to get the augmented parameters
augmented_params = params_augment_robust(params1, net1, net2)
params2 = augmented_params

In [ ]:
# Redefine simulate to accept net and Ne explicitly to handle the large network
def simulate_custom(net_target, params, input_amp, num_e):
    current_amp_nA = input_amp
    ac_currents = noise_current_ac(
        i_delay=t_on_global,
        i_dur=500.0,
        amp_n=0.0,
        amp_b=current_amp_nA,
        delta_t=dt_global,
        t_max=t_max_global,
        spect=jnp.array([120.0])
    )

    net_target.delete_stimuli()
    net_target.delete_recordings()

    # Stimulate a subset of E cells (scaling with network size)
    data_stimuli = net_target.cell([ik for ik in range(0, num_e, 2)]).branch(1).loc(0.0).data_stimulate(ac_currents)

    # Record from all cells
    # Fix for generator len error
    net_target.cell(list(range(len(list(net_target.cells))))).branch(0).loc(0.0).record()

    # Integrate
    traces = jx.integrate(net_target, params=params, data_stimuli=data_stimuli, checkpoint_lengths=checkpoints)
    return traces

def trace_vis_tfr(traces, label_psd=None, interval1=(1, 500), interval2=(501, 1000), dt_trace=0.1, savename=None):
    """
    Visualizes the traces with optional saving.
    """
    # 2. Raster
    spike_threshold = -20.0
    num_neurons = traces.shape[0]

    downsampling_factor = int(jnp.ceil(1.0 / dt_trace))
    if downsampling_factor <= 0:
        downsampling_factor = 1

    neuron_traces_data = traces[:, 1:] # Shape (num_neurons, original_num_timepoints)
    original_num_timepoints = neuron_traces_data.shape[1]

    num_timepoints_raster = (original_num_timepoints + downsampling_factor - 1) // downsampling_factor
    spike_image = jnp.zeros((num_neurons, num_timepoints_raster), dtype=jnp.float32)

    for i in range(num_neurons):
        neuron_trace = traces[i, 1:]
        neuron_trace_down = neuron_trace[::downsampling_factor]
        spikes = (neuron_trace_down[:-1] < spike_threshold) & (neuron_trace_down[1:] >= spike_threshold)
        spike_indices = jnp.where(spikes)[0]
        spike_image = spike_image.at[i, spike_indices].set(1.0)

    # Cumulative spiking
    sigma_samples = 1
    k_size_gauss = int(4 * sigma_samples + 1)
    x_grid_gauss = jnp.linspace(-3.0 * sigma_samples, 3.0 * sigma_samples, k_size_gauss)
    kernel_1d_gauss = jnp.exp(-x_grid_gauss**2 / (2 * sigma_samples**2))
    kernel_1d_gauss = kernel_1d_gauss / jnp.sum(kernel_1d_gauss) # Normalize
    kernel_2d_gauss = kernel_1d_gauss[None, :]
    spike_image_kernel = jax.scipy.signal.convolve2d(spike_image, kernel_2d_gauss, mode='same')
    cumulative_spiking = jnp.sum(spike_image_kernel, axis=0)

    # 3. Spectrogram (Summed across neurons)
    fs = 1000.0 / dt_trace
    nperseg = int(200.0 * fs / 1000.0) # 200ms window
    noverlap = int(nperseg * 0.99)

    # Get raw data (exclude V_init)
    traces_data = traces[:, 1:]

    # Extend traces with mirroring
    traces_extended = extend_traces_for_spectrogram(traces_data, nperseg)

    # Compute summed spectrogram
    f, t, Sxx_sum = compute_summed_spectrogram(traces_extended, fs, nperseg, noverlap)

    # Normalize summed Sxx for plotting
    Sxx_plot = Sxx_sum

    # Limit freq axis to interest (e.g. 0-200Hz)
    f_lim_idx = f <= 200.0
    f_plot = f[f_lim_idx]
    Sxx_plot_lim = Sxx_plot[f_lim_idx, :]

    t_shift = (nperseg / 2) / fs
    t_ms = (t - t_shift) * 1000.0

    # 4. Plot
    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(3, 1, height_ratios=[1, 0.1, 1], hspace=0.15)

    axs = []
    axs.append(fig.add_subplot(gs[0]))
    axs.append(fig.add_subplot(gs[1], sharex=axs[0]))
    axs.append(fig.add_subplot(gs[2], sharex=axs[0]))

    # Raster
    t_max = traces.shape[1] * dt_trace
    axs[0].imshow(spike_image, aspect='auto', cmap='Greys', origin='lower', interpolation='none',
                  extent=[0, t_max, 0, spike_image.shape[0]])
    axs[0].set_title("Model Raster")
    axs[0].set_ylabel("Neuron Index")
    axs[0].tick_params(labelbottom=False)

    # Cumulative Spiking
    raster_time_axis = jnp.linspace(0, t_max, num_timepoints_raster)
    axs[1].plot(raster_time_axis, cumulative_spiking, color='black', linewidth=0.8)
    axs[1].set_ylabel("Spike Count")
    axs[1].tick_params(labelbottom=False)

    # Spectrogram
    vis_smoothed_spectrogram(axs[2], Sxx_plot_lim, [t_ms[0], t_ms[-1]], [f_plot[0], f_plot[-1]])

    axs[2].set_title("Summed Spectrogram (Population Power)")
    axs[2].set_ylabel("Frequency (Hz)")
    axs[2].set_xlabel("Time (ms)")
    axs[2].set_ylim(0, 200)

    # Add red dashed lines
    red_line_freqs = [3.0, 8.0, 12.0, 32.0, 90.0, 120.0]
    for freq in red_line_freqs:
        axs[2].axhline(y=freq, color='red', linestyle='--', linewidth=1.0, alpha=0.8)

    plt.tight_layout()
    if savename:
        plt.savefig(savename, format='svg')
    plt.show()

# Redefine the visualization functions to use the custom simulate
def test_net_vis_comp_psd_custom(net_target, params, input_scalar, label_psd, label_psd_2, num_e, interval1=(1, 500), interval2=(501, 1000), savename=None):
    # 1. Simulate
    traces = simulate_custom(net_target, params, input_scalar, num_e) # Shape (Cells, Time)

    # Helper to calculate PSD and stats for a specific interval
    def analyze_interval(traces_segment, dt):
        # PSD
        signal = jnp.mean(traces_segment, axis=0)
        N = signal.shape[-1]
        if N == 0:
            return jnp.zeros_like(global_psd_interval), 0.0

        fs = 1000.0 / dt
        signal_fft = jnp.fft.rfft(signal)
        freqs = jnp.fft.rfftfreq(N, d=dt/1000)
        psd_raw = jnp.abs(signal_fft)**2 / (N * fs)

        target_freqs = global_psd_interval
        interpolated_psd = jnp.interp(target_freqs, freqs, psd_raw)
        max_psd = jnp.max(interpolated_psd)
        prediction_psd = interpolated_psd / (max_psd + 1e-6)

        # Firing Rate
        threshold = -20.0
        spikes = (traces_segment[:, :-1] < threshold) & (traces_segment[:, 1:] >= threshold)
        count = jnp.sum(spikes)
        n_cells = traces_segment.shape[0]
        duration_sec = (traces_segment.shape[1] * dt) / 1000.0
        mean_fr = count / n_cells / duration_sec

        return prediction_psd, mean_fr

    # Indices for slicing
    idx_start1 = int(interval1[0] / dt_global)
    idx_end1 = int(interval1[1] / dt_global)
    idx_start2 = int(interval2[0] / dt_global)
    idx_end2 = int(interval2[1] / dt_global)

    # Analyze Intervals
    psd1, mean_fr1 = analyze_interval(traces[:, idx_start1:idx_end1], dt_global)
    psd2, mean_fr2 = analyze_interval(traces[:, idx_start2:idx_end2], dt_global)

    # Resize labels if necessary to match global_psd_interval (e.g. 40 vs 80 bins)
    if label_psd.shape[0] != global_psd_interval.shape[0]:
        # Interpolate label to match global_psd_interval
        old_x = jnp.linspace(global_psd_interval[0], global_psd_interval[-1], label_psd.shape[0])
        label_psd = jnp.interp(global_psd_interval, old_x, label_psd)

    if label_psd_2.shape[0] != global_psd_interval.shape[0]:
        old_x = jnp.linspace(global_psd_interval[0], global_psd_interval[-1], label_psd_2.shape[0])
        label_psd_2 = jnp.interp(global_psd_interval, old_x, label_psd_2)

    # Calculate Loss for Interval 1
    epsilon = 1e-6
    psd_loss1 = jnp.sum(jnp.square(label_psd * jnp.log((psd1 + epsilon) / (label_psd + epsilon))))
    # psd_loss2 = jnp.sum(jnp.square(label_psd_2 * jnp.log((psd2 + epsilon) / (label_psd_2 + epsilon))))

    # Firing rate penalty
    penalty_lower = jnp.exp(lower_c - mean_fr1)
    penalty_upper = jnp.exp(mean_fr1 - upper_c)
    firing_rate_penalty1 = penalty_lower + penalty_upper

    total_loss1 = psd_loss1 * psd_weight + firing_rate_weight * firing_rate_penalty1

    # Visualization
    fig, axs = plt.subplots(2, 1, figsize=(12, 6))

    # Upsample/Smooth for vis
    upsample_factor = 10
    smoothing_sigma = 1.5
    global_psd_interval_upsampled = jnp.linspace(
        global_psd_interval[0], global_psd_interval[-1],
        len(global_psd_interval) * upsample_factor
    )
    psd1_upsampled = jnp.interp(global_psd_interval_upsampled, global_psd_interval, psd1)
    psd1_smoothed = ndimage.gaussian_filter(psd1_upsampled, sigma=smoothing_sigma)
    psd2_upsampled = jnp.interp(global_psd_interval_upsampled, global_psd_interval, psd2)
    psd2_smoothed = ndimage.gaussian_filter(psd2_upsampled, sigma=smoothing_sigma)

    # Plot
    axs[0].plot(global_psd_interval, label_psd, label='Target (Label_pre)', linestyle='--', color='black')
    axs[0].plot(global_psd_interval_upsampled, psd1_smoothed, label=f'Prediction ({interval1}ms) Smoothed', color='red')
    axs[0].set_title(f"Interval {interval1}ms - Total Loss: {total_loss1:.4f}")
    axs[0].legend()

    axs[1].plot(global_psd_interval, label_psd_2, label='Target (Label_stim)', linestyle='--', color='black')
    axs[1].plot(global_psd_interval_upsampled, psd2_smoothed, label=f'Prediction ({interval2}ms) Smoothed', color='blue')
    axs[1].set_title(f"Interval {interval2}ms")
    axs[1].legend()

    if savename: plt.savefig(savename, format='svg')
    plt.show()

def test_net_vis_comp_tfr_custom(net_target, params, input_scalar, num_e, label_psd=None, interval1=(1, 500), interval2=(501, 1000), savename=None):
    # 1. Simulate
    traces = simulate_custom(net_target, params, input_scalar, num_e)

    # Call the existing visualization logic for traces
    trace_vis_tfr(traces, label_psd, interval1, interval2, dt_trace=dt_global, savename=savename)

# Execute
Nm2 = 50
test_net_vis_comp_psd_custom(net2, params2, inputs[1], labels[0], labels[1], Nm2, savename="PSD_comp_10.svg")
test_net_vis_comp_tfr_custom(net2, params2, inputs[2]*1.0, Nm2, savename="TFR_raster_11.svg")

In [ ]:
def get_connectivity_matrix(net, param):
    nodes = list(net.cells)
    N = len(nodes)
    mat = np.zeros((N, N))

    if param in net.edges.columns:
        df = net.edges
        # Filter for rows where the parameter is not NaN
        valid = df[param].notna()

        if not valid.any():
            return mat

        # Get node indices
        pre_nodes = df.loc[valid, 'pre_index'].values.astype(int)
        post_nodes = df.loc[valid, 'post_index'].values.astype(int)

        # Map node indices to cell indices
        node_to_cell = net.nodes['local_cell_index'].values

        # Safety check for indices
        valid_mask = (pre_nodes < len(node_to_cell)) & (post_nodes < len(node_to_cell))

        pre = node_to_cell[pre_nodes[valid_mask]]
        post = node_to_cell[post_nodes[valid_mask]]
        vals = df.loc[valid, param].values[valid_mask]

        mat[pre, post] = vals

    return mat

# Extract matrices
W1 = get_connectivity_matrix(net1, 'gAMPA')
W2 = get_connectivity_matrix(net2, 'gAMPA')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(W1, cmap='viridis', aspect='auto', interpolation='nearest')
axes[0].set_title(f"Net1 gAMPA (Shape: {W1.shape})")
axes[0].set_xlabel("Postsynaptic Cell")
axes[0].set_ylabel("Presynaptic Cell")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(W2, cmap='viridis', aspect='auto', interpolation='nearest')
axes[1].set_title(f"Net2 gAMPA (Upscaled, Shape: {W2.shape})")
axes[1].set_xlabel("Postsynaptic Cell")
axes[1].set_ylabel("Presynaptic Cell")
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
test_net_vis_comp_psd(net2, params2, inputs[1], labels[0], labels[1], interval1=(1, 500), interval2=(501, 1000), savename="PSD_comp_1.svg")
test_net_vis_comp_tfr(net2, params2, inputs[1], label_psd=None, interval1=(1, 500), interval2=(501, 1000), savename="TFR_raster_1.svg")

In [ ]:
def trace_vis_tfr(traces, label_psd=None, interval1=(1, 500), interval2=(501, 1000), dt_trace=0.1, savename=None):
    """
    Visualizes the traces with optional saving.
    """
    # 2. Raster
    spike_threshold = -20.0
    num_neurons = traces.shape[0]

    downsampling_factor = int(jnp.ceil(1.0 / dt_trace))
    if downsampling_factor <= 0:
        downsampling_factor = 1

    neuron_traces_data = traces[:, 1:] # Shape (num_neurons, original_num_timepoints)
    original_num_timepoints = neuron_traces_data.shape[1]

    num_timepoints_raster = (original_num_timepoints + downsampling_factor - 1) // downsampling_factor
    spike_image = jnp.zeros((num_neurons, num_timepoints_raster), dtype=jnp.float32)

    for i in range(num_neurons):
        neuron_trace = traces[i, 1:]
        neuron_trace_down = neuron_trace[::downsampling_factor]
        spikes = (neuron_trace_down[:-1] < spike_threshold) & (neuron_trace_down[1:] >= spike_threshold)
        spike_indices = jnp.where(spikes)[0]
        spike_image = spike_image.at[i, spike_indices].set(1.0)

    # Cumulative spiking
    sigma_samples = 1
    k_size_gauss = int(4 * sigma_samples + 1)
    x_grid_gauss = jnp.linspace(-3.0 * sigma_samples, 3.0 * sigma_samples, k_size_gauss)
    kernel_1d_gauss = jnp.exp(-x_grid_gauss**2 / (2 * sigma_samples**2))
    kernel_1d_gauss = kernel_1d_gauss / jnp.sum(kernel_1d_gauss) # Normalize
    kernel_2d_gauss = kernel_1d_gauss[None, :]
    spike_image_kernel = jax.scipy.signal.convolve2d(spike_image, kernel_2d_gauss, mode='same')
    cumulative_spiking = jnp.sum(spike_image_kernel, axis=0)

    # 3. Spectrogram (Summed across neurons)
    fs = 1000.0 / dt_trace
    nperseg = int(200.0 * fs / 1000.0) # 200ms window
    noverlap = int(nperseg * 0.99)

    # Get raw data (exclude V_init)
    traces_data = traces[:, 1:]

    # Extend traces with mirroring
    traces_extended = extend_traces_for_spectrogram(traces_data, nperseg)

    # Compute summed spectrogram
    f, t, Sxx_sum = compute_summed_spectrogram(traces_extended, fs, nperseg, noverlap)

    # Normalize summed Sxx for plotting
    Sxx_plot = Sxx_sum #np.sqrt(Sxx_sum)

    # Limit freq axis to interest (e.g. 0-100Hz)
    f_lim_idx = f <= 100.0
    f_plot = f[f_lim_idx]
    Sxx_plot_lim = Sxx_plot[f_lim_idx, :]

    t_shift = (nperseg / 2) / fs
    t_ms = (t - t_shift) * 1000.0

    # 4. Plot
    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(3, 1, height_ratios=[1, 0.1, 1], hspace=0.15)

    axs = []
    axs.append(fig.add_subplot(gs[0]))
    axs.append(fig.add_subplot(gs[1], sharex=axs[0]))
    axs.append(fig.add_subplot(gs[2], sharex=axs[0]))

    # Raster
    t_max = traces.shape[1] * dt_trace
    axs[0].imshow(spike_image, aspect='auto', cmap='Greys', origin='lower', interpolation='none',
                  extent=[0, t_max, 0, spike_image.shape[0]])
    axs[0].set_title("Model Raster")
    axs[0].set_ylabel("Neuron Index")
    axs[0].tick_params(labelbottom=False)

    # Cumulative Spiking
    raster_time_axis = jnp.linspace(0, t_max, num_timepoints_raster)
    axs[1].plot(raster_time_axis, cumulative_spiking, color='black', linewidth=0.8)
    axs[1].set_ylabel("Spike Count")
    axs[1].tick_params(labelbottom=False)

    # Spectrogram
    vis_smoothed_spectrogram(axs[2], Sxx_plot_lim, [t_ms[0], t_ms[-1]], [f_plot[0], f_plot[-1]])

    axs[2].set_title("Summed Spectrogram (Population Power)")
    axs[2].set_ylabel("Frequency (Hz)")
    axs[2].set_xlabel("Time (ms)")
    axs[2].set_ylim(0, 100)

    # Add red dashed lines
    red_line_freqs = [3.0, 8.0, 12.0, 32.0, 90.0]
    for freq in red_line_freqs:
        axs[2].axhline(y=freq, color='red', linestyle='--', linewidth=1.0, alpha=0.8)

    plt.tight_layout()
    if savename:
        plt.savefig(savename, format='svg')
    plt.show()

def trace_vis_psd(traces, dt=0.1, f_min=1.0, f_max=100.0, savename=None):
    """
    Visualizes the Power Spectral Density (PSD) of the mean trace.
    """
    # Mean trace
    signal = jnp.mean(traces[:, 1:], axis=0)
    N = signal.shape[-1]
    fs = 1000.0 / dt

    # PSD
    signal_fft = jnp.fft.rfft(signal)
    freqs = jnp.fft.rfftfreq(N, d=dt/1000)
    psd_raw = jnp.abs(signal_fft)**2 / (N * fs)

    # Plotting
    plt.figure(figsize=(8, 5))
    mask = (freqs >= f_min) & (freqs <= f_max)
    plt.plot(freqs[mask], psd_raw[mask], color='blue')
    plt.title("Population PSD")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Power")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if savename:
        plt.savefig(savename, format='svg')
    plt.show()

dt_global = 0.1
t_max_vs = 1500.0
t_onset_vs = 500.0
t_dur_vs = 500.0
single_input_amp = inputs[0]*1.0

levels = 2
time_points_vs = t_max_vs / dt_global + 1
checkpoints_vs = [int(np.ceil(time_points_vs**(1/levels))) for _ in range(levels)]

ac_current = noise_current_ac(
    i_delay=t_onset_vs,
    i_dur=t_dur_vs,
    amp_n=0.0,
    amp_b=single_input_amp,
    delta_t=dt_global,
    t_max=t_max_vs,
    spect=jnp.array([120.0])
)

sim_current = ac_current

this_params = net2.get_parameters()


net2.delete_stimuli()
net2.delete_recordings()
num_posts_to_select = int(Ne2/2)
posts_pool_indices = jnp.concat([jnp.arange(0, Ne2)])
key_conn = jax.random.PRNGKey(5678)
selected_post_indices = np.array(jax.random.choice(key_conn, posts_pool_indices, shape=(num_posts_to_select,), replace=False))

data_stimuli_single_run = net2.cell(selected_post_indices).branch(1).loc(0.0).data_stimulate(sim_current) # Changed loc(1.0) to loc(0.0)
net2.cell([i for i in range(Ne2+Nig2+Nil2)]).branch(0).loc(0.0).record()

print(f"Running simulation with input amplitude: {single_input_amp:.2f} nA...")
recorded_voltages_ag = jx.integrate(net2, params=this_params, data_stimuli=data_stimuli_single_run, checkpoint_lengths=checkpoints_vs)
print(f"Simulation completed. Recorded voltages shape: {recorded_voltages_ag.shape}")
time_axis = jnp.arange(20, t_max_vs, dt_global)

trace_vis_tfr(recorded_voltages_ag, savename="trace_vis_tfr4.svg")

In [ ]:
import matplotlib.pyplot as plt
import jax.numpy as jnp
from scipy import ndimage

def trace_comp_psd(traces, input_scalar, label_psd, label_psd_2, interval1=(1, 500), interval2=(501, 1000), savename=None):
    """
    Visualizes the PSD prediction and loss for pre-calculated traces over two specific intervals.
    """
    # Helper to calculate PSD and stats for a specific interval
    def analyze_interval(traces_segment, dt):
        # PSD
        signal = jnp.mean(traces_segment, axis=0)
        N = signal.shape[-1]
        if N == 0:
            return jnp.zeros_like(global_psd_interval), 0.0

        fs = 1000.0 / dt
        signal_fft = jnp.fft.rfft(signal)
        freqs = jnp.fft.rfftfreq(N, d=dt/1000)
        psd_raw = jnp.abs(signal_fft)**2 / (N * fs)

        target_freqs = global_psd_interval
        interpolated_psd = jnp.interp(target_freqs, freqs, psd_raw)
        max_psd = jnp.max(interpolated_psd)
        prediction_psd = interpolated_psd / (max_psd + 1e-6)

        # Firing Rate
        threshold = -20.0
        spikes = (traces_segment[:, :-1] < threshold) & (traces_segment[:, 1:] >= threshold)
        count = jnp.sum(spikes)
        n_cells = traces_segment.shape[0]
        duration_sec = (traces_segment.shape[1] * dt) / 1000.0
        mean_fr = count / n_cells / duration_sec

        return prediction_psd, mean_fr

    # Indices for slicing
    idx_start1 = int(interval1[0] / dt_global)
    idx_end1 = int(interval1[1] / dt_global)
    idx_start2 = int(interval2[0] / dt_global)
    idx_end2 = int(interval2[1] / dt_global)

    # Analyze Intervals
    psd1, mean_fr1 = analyze_interval(traces[:, idx_start1:idx_end1], dt_global)
    psd2, mean_fr2 = analyze_interval(traces[:, idx_start2:idx_end2], dt_global)

    # Calculate Loss for Interval 1
    epsilon = 1e-6
    psd_loss1 = jnp.sum(jnp.square(label_psd * jnp.log((psd1 + epsilon) / (label_psd + epsilon))))
    psd_loss2 = jnp.sum(jnp.square(label_psd_2 * jnp.log((psd2 + epsilon) / (label_psd_2 + epsilon))))

    # Firing rate penalty approximation for single scalar mean FR
    penalty_lower = jnp.exp(lower_c - mean_fr1)
    penalty_upper = jnp.exp(mean_fr1 - upper_c)
    firing_rate_penalty1 = penalty_lower + penalty_upper

    total_loss1 = psd_loss1 * psd_weight + firing_rate_weight * firing_rate_penalty1

    # Visualization
    fig, axs = plt.subplots(2, 1, figsize=(12, 6))

    # Upsample and smoothen prediction lines
    upsample_factor = 10
    smoothing_sigma = 1.0

    global_psd_interval_upsampled = jnp.linspace(
        global_psd_interval[0], global_psd_interval[-1],
        len(global_psd_interval) * upsample_factor
    )

    # Smooth first
    psd1_smoothed = ndimage.gaussian_filter(psd1, sigma=smoothing_sigma)
    psd2_smoothed = ndimage.gaussian_filter(psd2, sigma=smoothing_sigma)

    # Then upsample (interpolate)
    psd1_vis = jnp.interp(global_psd_interval_upsampled, global_psd_interval, psd1_smoothed)
    psd2_vis = jnp.interp(global_psd_interval_upsampled, global_psd_interval, psd2_smoothed)

    # Scaling helper
    def scale_vis(arr):
        min_val = jnp.min(arr)
        max_val = jnp.max(arr)
        denom = max_val - min_val
        denom = jnp.where(denom == 0, 1.0, denom)
        arr_scaled = (arr - min_val) / denom
        return arr_scaled * (1.0 - 1e-6) + 1e-6

    # Apply scaling for visualization
    psd1_vis = scale_vis(psd1_vis)
    psd2_vis = scale_vis(psd2_vis)


    # Subplot 211: Interval 1
    axs[0].plot(global_psd_interval, label_psd, label='Target (Label_pre)', linestyle='--', color='black')
    axs[0].plot(global_psd_interval_upsampled, psd1_vis, label=f'Prediction ({interval1}ms) Smoothed', color='red')
    axs[0].set_title(f"Interval {interval1}ms - Total Loss: {total_loss1:.4f}")
    axs[0].set_xlabel("Frequency (Hz)")
    axs[0].set_ylabel("Normalized Power")
    axs[0].legend()

    stats_text1 = (f"Total Loss: {total_loss1:.4f}\n"
                   f"PSD Loss: {psd_loss1*psd_weight:.4f}\n"
                   f"FR Penalty: {firing_rate_penalty1*firing_rate_weight:.4f}\n"
                   f"Mean FR: {mean_fr1:.2f} Hz")
    axs[0].text(0.05, 0.95, stats_text1,
                transform=axs[0].transAxes, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

    # Subplot 212: Interval 2
    axs[1].plot(global_psd_interval, label_psd_2, label='Target (Label_stim)', linestyle='--', color='black')
    axs[1].plot(global_psd_interval_upsampled, psd2_vis, label=f'Prediction ({interval2}ms) Smoothed', color='blue')
    axs[1].set_title(f"Interval {interval2}ms")
    axs[1].set_xlabel("Frequency (Hz)")
    axs[1].set_ylabel("Normalized Power")
    axs[1].legend()
    axs[1].text(0.05, 0.95, f"Mean FR: {mean_fr2:.2f} Hz",
                transform=axs[1].transAxes, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

    plt.tight_layout()
    if savename:
        plt.savefig(savename, format='svg')
    plt.show()

    return total_loss1

trace_comp_psd(recorded_voltages_ag, inputs[1], labels[0], labels[1], interval1=(1, 500), interval2=(501, 1000), savename="PSD_comp_ag1.svg")

# End